# AURA — AI Research & Decision Agent

AURA is a multi-agent AI system designed to transform scientific and technical questions into evidence-backed research, verified insights, and actionable implementation plans.

## MVP Goal

The MVP should:

1. Accept a scientific or technical research question
2. Decompose it into research tasks
3. Discover relevant papers, datasets, code, and documentation
4. Extract useful evidence
5. Retrieve relevant evidence using RAG
6. Verify important claims
7. Compare methods and limitations
8. Generate a cited research report
9. Produce an implementation roadmap

## Initial MVP Flow

User Question
→ Task Planner
→ Research Agent
→ Evidence Extraction
→ RAG
→ Verification
→ Analysis
→ Decision & Roadmap
→ Final Report

In [1]:
!pip install google-genai python-dotenv pydantic requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 29.2 MB/s  0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.0
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.33.2
    Uninstalling pydantic_core-2.33.2:
      Successfully uninstalled pydantic_core-2.33.2
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.11.10
    Uninstalling pydantic-2.11.10:
      Successfully uninstalled pydantic-2.11.10
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.34.0
    Not uninstalling google-auth at /toolkit-cache/2.6.1/python3.13/kernel-libs/lib/python3.13/site-packages, outside environment /root/venv
    Can't uninstall 'google-auth'. No files were found to uninstall.
   ━━━

In [23]:
import os
import json
from typing import List
from pydantic import BaseModel, Field
import time

In [5]:
api_key = os.getenv("GEMINI_API_KEY")

if api_key:
    print("✅ Gemini API key loaded successfully.")
else:
    print("❌ Gemini API key not found.")

✅ Gemini API key loaded successfully.


In [7]:
from google import genai

client = genai.Client(api_key=api_key)

print("✅ Gemini client created successfully.")

✅ Gemini client created successfully.


In [11]:
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="""
    Introduce yourself in one short sentence as the reasoning engine of
    AURA, an AI Research & Decision Agent.
    """
)

print(response.text)

I am the reasoning engine behind AURA, powering intelligent research and strategic decision-making.


In [109]:
print("Available Gemini models:\n")

for model in client.models.list():
    if "flash" in model.name.lower():
        print(model.name)

Available Gemini models:

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


In [111]:
FAST_MODEL = "gemini-3.5-flash-lite"
DEFAULT_MODEL = "gemini-3.5-flash"
REASONING_MODEL = "gemini-3.8-flash"

print("Fast model:", FAST_MODEL)
print("Default model:", DEFAULT_MODEL)
print("Reasoning model:", REASONING_MODEL)

Fast model: gemini-3.5-flash-lite
Default model: gemini-3.5-flash
Reasoning model: gemini-3.8-flash


In [170]:
def generate_with_fallback(
    prompt: str,
    preferred_model: str = DEFAULT_MODEL
):
    models_to_try = [
        preferred_model,
        DEFAULT_MODEL,
        "gemini-3.6-flash",
        "gemini-3.7-flash",
        FAST_MODEL
    ]

    # Remove duplicates while preserving order
    models_to_try = list(dict.fromkeys(models_to_try))

    last_error = None

    for model_name in models_to_try:
        try:
            print(f"Trying model: {model_name}")

            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )

            print(f"✅ Success with: {model_name}")
            return response

        except Exception as e:
            last_error = e
            error_text = str(e)

            recoverable_errors = [
                "503",
                "UNAVAILABLE",
                "429",
                "RESOURCE_EXHAUSTED",
                "quota"
            ]

            if any(
                error.lower() in error_text.lower()
                for error in recoverable_errors
            ):
                print(
                    f"⚠️ {model_name} unavailable or quota exhausted. "
                    f"Trying another model..."
                )
                continue

            raise e

    raise RuntimeError(
        f"All Gemini models failed. Last error: {last_error}"
    )

## Phase 2 — Task Planner Agent

The Task Planner Agent receives a scientific or technical research question
and decomposes it into structured research tasks.

Its responsibilities are:

- understand the user's main goal
- identify the scientific or technical domain
- identify constraints
- break the problem into research sub-tasks
- prepare search queries for later research agents

In [13]:
class ResearchTask(BaseModel):
    id: int
    task_type: str
    query: str
    purpose: str


class ResearchPlan(BaseModel):
    goal: str
    domain: str
    constraints: List[str] = Field(default_factory=list)
    tasks: List[ResearchTask]

In [15]:
PLANNER_PROMPT = """
You are the Task Planner Agent of AURA,
an AI Research & Decision Agent.

Your job is to analyze a scientific or technical research question
and create a structured research plan.

You must:

1. Understand the user's main goal.
2. Identify the scientific or technical domain.
3. Identify important constraints if they exist.
4. Break the problem into clear research tasks.
5. Create useful search queries for each task.

Possible task types include:

- paper_search
- dataset_search
- code_search
- documentation_search
- method_comparison
- limitation_analysis
- implementation_planning

Do not answer the research question itself.

Your job is only to PLAN the research.
"""

In [27]:
def create_research_plan(user_question: str):
    prompt = f"""
{PLANNER_PROMPT}

USER RESEARCH QUESTION:

{user_question}

Return ONLY valid JSON using exactly this structure:

{{
    "goal": "main research goal",
    "domain": "scientific or technical domain",
    "constraints": [
        "constraint 1"
    ],
    "tasks": [
        {{
            "id": 1,
            "task_type": "paper_search",
            "query": "search query",
            "purpose": "why this research task is needed"
        }}
    ]
}}
"""

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model="gemini-3.8-flash",
                contents=prompt
            )
            break

        except Exception as e:
            if attempt < 2:
                print(
                    f"Gemini temporarily unavailable. "
                    f"Retrying... ({attempt + 1}/3)"
                )
                time.sleep(10)
            else:
                raise e

    clean_text = response.text.strip()

    if clean_text.startswith("```json"):
        clean_text = clean_text[7:]

    if clean_text.startswith("```"):
        clean_text = clean_text[3:]

    if clean_text.endswith("```"):
        clean_text = clean_text[:-3]

    plan_dict = json.loads(clean_text.strip())

    validated_plan = ResearchPlan(**plan_dict)

    return validated_plan

In [29]:
question = """
I want to build a solar flare prediction system.

Find relevant scientific papers, datasets, machine learning methods,
existing code implementations, and technical documentation.

Compare the approaches, identify their limitations,
and propose an implementation plan.
"""

In [31]:
plan = create_research_plan(question)

plan

Gemini temporarily unavailable. Retrying... (1/3)
Gemini temporarily unavailable. Retrying... (2/3)


ResearchPlan(goal='Design and outline an end-to-end solar flare prediction system using machine learning by reviewing literature, identifying datasets and open-source implementations, evaluating modeling techniques, and structuring an actionable deployment plan.', domain='Heliophysics / Space Weather Forecasting / Applied Machine Learning', constraints=['Address severe class imbalance inherent to high-energy solar flare events (M- and X-class)', 'Rely on publicly accessible satellite data streams (e.g., SDO/HMI, GOES)', 'Focus on operational operational metrics (e.g., True Skill Statistic, Heidke Skill Score) rather than standard accuracy'], tasks=[ResearchTask(id=1, task_type='paper_search', query='solar flare prediction machine learning deep learning SDO HMI magnetograms review', purpose='Identify state-of-the-art machine learning architectures and methodologies applied to solar flare forecasting.'), ResearchTask(id=2, task_type='dataset_search', query='solar flare benchmark dataset 

In [33]:
print(json.dumps(
    plan.model_dump(),
    indent=2,
    ensure_ascii=False
))

{
  "goal": "Design and outline an end-to-end solar flare prediction system using machine learning by reviewing literature, identifying datasets and open-source implementations, evaluating modeling techniques, and structuring an actionable deployment plan.",
  "domain": "Heliophysics / Space Weather Forecasting / Applied Machine Learning",
  "constraints": [
    "Address severe class imbalance inherent to high-energy solar flare events (M- and X-class)",
    "Rely on publicly accessible satellite data streams (e.g., SDO/HMI, GOES)",
    "Focus on operational operational metrics (e.g., True Skill Statistic, Heidke Skill Score) rather than standard accuracy"
  ],
  "tasks": [
    {
      "id": 1,
      "task_type": "paper_search",
      "query": "solar flare prediction machine learning deep learning SDO HMI magnetograms review",
      "purpose": "Identify state-of-the-art machine learning architectures and methodologies applied to solar flare forecasting."
    },
    {
      "id": 2,
   

## Phase 3 — Research & Discovery Agent

The Research & Discovery Agent executes research tasks created by the
Task Planner.

The first MVP capability focuses on scientific paper discovery.

Responsibilities:

- receive a paper-search task
- query a real academic source
- retrieve paper metadata
- preserve source URLs and identifiers
- return structured research sources

### Academic Sources

- **OpenAlex** — Primary academic source for the MVP ✅
- **Semantic Scholar** — Optional future integration; authenticated API access is currently unavailable.

AURA is designed so academic search providers can be replaced or extended without changing the downstream research pipeline.

In [35]:
class PaperSource(BaseModel):
    title: str
    authors: List[str] = Field(default_factory=list)
    year: int | None = None
    abstract: str | None = None
    url: str | None = None
    citation_count: int | None = None
    paper_id: str | None = None

### Academic Source 2 — OpenAlex

OpenAlex is used as the primary academic search source for the MVP
while the Semantic Scholar API key is pending.

In [56]:
def search_openalex(query: str, limit: int = 5):
    url = "https://api.openalex.org/works"

    params = {
        "search": query,
        "per-page": limit
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    return response.json()

In [58]:
raw_openalex = search_openalex(
    query=paper_query,
    limit=5
)

raw_openalex

{'meta': {'count': 243,
  'db_response_time_ms': 230,
  'page': 1,
  'per_page': 5,
  'groups_count': None,
  'x_query': {'oql': 'works where full text has (solar flare prediction machine learning deep learning SDO HMI magnetograms review)',
   'oqo': {'get_rows': 'works',
    'filter_rows': [{'column_id': 'fulltext.search',
      'value': 'solar flare prediction machine learning deep learning SDO HMI magnetograms review',
      'operator': 'has'}]},
   'url': '/works?filter=fulltext.search:solar flare prediction machine learning deep learning SDO HMI magnetograms review&per_page=5'},
  'cost_usd': 0.001},
 'results': [{'id': 'https://openalex.org/W2920947047',
   'doi': 'https://doi.org/10.1029/2018sw002061',
   'title': 'The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting',
   'display_name': 'The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting',
   'relevance_score': 606.61646,
   'publication_year': 2019,
   'publication_date': 

In [48]:
import time
import requests

def search_semantic_scholar(query: str, limit: int = 5):
    url = "https://api.semanticscholar.org/graph/v1/paper/search"

    params = {
        "query": query,
        "limit": limit,
        "fields": "paperId,title,authors,year,abstract,url,citationCount"
    }

    for attempt in range(3):
        response = requests.get(
            url,
            params=params,
            timeout=30
        )

        if response.status_code == 200:
            return response.json()

        if response.status_code == 429:
            if attempt < 2:
                wait_time = 15 * (attempt + 1)
                print(
                    f"Semantic Scholar rate limit reached. "
                    f"Retrying in {wait_time} seconds..."
                )
                time.sleep(wait_time)
                continue

        response.raise_for_status()

In [50]:
paper_tasks = [
    task
    for task in plan.tasks
    if task.task_type == "paper_search"
]

paper_tasks

[ResearchTask(id=1, task_type='paper_search', query='solar flare prediction machine learning deep learning SDO HMI magnetograms review', purpose='Identify state-of-the-art machine learning architectures and methodologies applied to solar flare forecasting.')]

In [52]:
paper_query = paper_tasks[0].query

print("Search query:")
print(paper_query)

Search query:
solar flare prediction machine learning deep learning SDO HMI magnetograms review


### OpenAlex Search

OpenAlex is currently used as the primary academic search source
for the AURA MVP.

In [60]:
def search_openalex(query: str, limit: int = 5):
    url = "https://api.openalex.org/works"

    params = {
        "search": query,
        "per-page": limit
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    return response.json()

In [62]:
raw_openalex = search_openalex(
    query=paper_query,
    limit=5
)

raw_openalex

{'meta': {'count': 243,
  'db_response_time_ms': 371,
  'page': 1,
  'per_page': 5,
  'groups_count': None,
  'x_query': {'oql': 'works where full text has (solar flare prediction machine learning deep learning SDO HMI magnetograms review)',
   'oqo': {'get_rows': 'works',
    'filter_rows': [{'column_id': 'fulltext.search',
      'value': 'solar flare prediction machine learning deep learning SDO HMI magnetograms review',
      'operator': 'has'}]},
   'url': '/works?filter=fulltext.search:solar flare prediction machine learning deep learning SDO HMI magnetograms review&per_page=5'},
  'cost_usd': 0.001},
 'results': [{'id': 'https://openalex.org/W2920947047',
   'doi': 'https://doi.org/10.1029/2018sw002061',
   'title': 'The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting',
   'display_name': 'The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting',
   'relevance_score': 606.61646,
   'publication_year': 2019,
   'publication_date': 

In [64]:
def reconstruct_abstract(inverted_index):
    if not inverted_index:
        return None

    positions = []

    for word, indexes in inverted_index.items():
        for index in indexes:
            positions.append((index, word))

    positions.sort(key=lambda item: item[0])

    abstract = " ".join(
        word for _, word in positions
    )

    return abstract

In [66]:
def normalize_openalex_papers(search_result):
    papers = []

    for work in search_result.get("results", []):
        authors = []

        for authorship in work.get("authorships", []):
            author = authorship.get("author", {})
            name = author.get("display_name")

            if name:
                authors.append(name)

        abstract = reconstruct_abstract(
            work.get("abstract_inverted_index")
        )

        paper = PaperSource(
            title=work.get("display_name", ""),
            authors=authors,
            year=work.get("publication_year"),
            abstract=abstract,
            url=work.get("doi") or work.get("id"),
            citation_count=work.get("cited_by_count"),
            paper_id=work.get("id")
        )

        papers.append(paper)

    return papers

In [68]:
papers = normalize_openalex_papers(raw_openalex)

print(f"✅ {len(papers)} papers normalized.")

✅ 5 papers normalized.


In [70]:
for index, paper in enumerate(papers, start=1):
    print("=" * 80)
    print(f"Paper #{index}")
    print(f"Title: {paper.title}")
    print(f"Year: {paper.year}")
    print(f"Authors: {', '.join(paper.authors[:5])}")
    print(f"Citations: {paper.citation_count}")
    print(f"URL: {paper.url}")

    if paper.abstract:
        print(f"Abstract: {paper.abstract[:500]}...")

    print()

Paper #1
Title: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
Year: 2019
Authors: Enrico Camporeale
Citations: 415
URL: https://doi.org/10.1029/2018sw002061
Abstract: Abstract The numerous recent breakthroughs in machine learning make imperative to carefully ponder how the scientific community can benefit from a technology that, although not necessarily new, is today living its golden age. This Grand Challenge review paper is focused on the present and future role of machine learning in Space Weather. The purpose is twofold. On one hand, we will discuss previous works that use machine learning for Space Weather forecasting, focusing in particular on the few a...

Paper #2
Title: Solar Flare Intensity Prediction With Machine Learning Models
Year: 2020
Authors: Zhenbang Jiao, Hu Sun, Xiantong Wang, W. B. Manchester, T. I. Gombosi
Citations: 59
URL: https://doi.org/10.1029/2020sw002440
Abstract: Abstract We develop a mixed long short‐term memory (LSTM) reg

## Phase 4 — Evidence Extraction Agent

The Evidence Extraction Agent converts retrieved scientific papers
into structured, traceable research evidence.

For the MVP, extraction is based on paper abstracts.

Responsibilities:

- identify the research focus
- extract methods mentioned in the source
- identify datasets or data sources
- extract reported findings
- identify stated limitations when available
- preserve provenance to the original paper
- avoid unsupported claims

In [72]:
class EvidenceItem(BaseModel):
    paper_id: str | None = None
    source_title: str
    source_url: str | None = None

    research_focus: str
    methods: List[str] = Field(default_factory=list)
    datasets: List[str] = Field(default_factory=list)
    findings: List[str] = Field(default_factory=list)
    limitations: List[str] = Field(default_factory=list)

    evidence_summary: str

In [74]:
def clean_json_response(text: str):
    clean_text = text.strip()

    if clean_text.startswith("```json"):
        clean_text = clean_text[7:]
    elif clean_text.startswith("```"):
        clean_text = clean_text[3:]

    if clean_text.endswith("```"):
        clean_text = clean_text[:-3]

    return json.loads(clean_text.strip())

In [76]:
def extract_evidence(paper: PaperSource):
    if not paper.abstract:
        return None

    prompt = f"""
You are the Evidence Extraction Agent of AURA.

Your task is to extract structured research evidence ONLY from the
paper information provided below.

Do not use outside knowledge.
Do not invent methods, datasets, findings, or limitations.

If something is not explicitly supported by the abstract,
return an empty list for that field.

PAPER TITLE:
{paper.title}

ABSTRACT:
{paper.abstract}

Return ONLY valid JSON with exactly this structure:

{{
    "paper_id": "{paper.paper_id}",
    "source_title": "{paper.title}",
    "source_url": "{paper.url}",
    "research_focus": "short description of what the paper studies",
    "methods": [],
    "datasets": [],
    "findings": [],
    "limitations": [],
    "evidence_summary": "short evidence-grounded summary"
}}
"""

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model="gemini-3.8-flash",
                contents=prompt
            )
            break

        except Exception as e:
            if attempt < 2:
                wait_time = 10 * (attempt + 1)

                print(
                    f"Gemini temporarily unavailable. "
                    f"Retrying in {wait_time} seconds..."
                )

                time.sleep(wait_time)
            else:
                raise e

    evidence_dict = clean_json_response(response.text)

    return EvidenceItem(**evidence_dict)

In [78]:
test_evidence = extract_evidence(papers[1])

test_evidence

EvidenceItem(paper_id='https://openalex.org/W3024706845', source_title='Solar Flare Intensity Prediction With Machine Learning Models', source_url='https://doi.org/10.1029/2020sw002440', research_focus='Predicting the maximum solar flare intensity and solar flare classifications using machine learning models across various advance time windows.', methods=['Mixed long short-term memory (LSTM) regression model', 'Classification models built on top of the LSTM regression model'], datasets=['Space-Weather Helioseismic and Magnetic Imager Active Region Patch (SHARP) parameters / Helioseismic and Magnetic Imager (HMI) Active Region Patch (HARP) data', 'Geostationary Operational Environmental Satellites (GOES) data set'], findings=['The mixed LSTM regression model provides detailed information on the exact maximum flux level (intensity) for each flare occurrence rather than just class labels.', 'Classification models built on top of the regression model achieved better results in solar flare 

In [80]:
print(
    json.dumps(
        test_evidence.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)

{
  "paper_id": "https://openalex.org/W3024706845",
  "source_title": "Solar Flare Intensity Prediction With Machine Learning Models",
  "source_url": "https://doi.org/10.1029/2020sw002440",
  "research_focus": "Predicting the maximum solar flare intensity and solar flare classifications using machine learning models across various advance time windows.",
  "methods": [
    "Mixed long short-term memory (LSTM) regression model",
    "Classification models built on top of the LSTM regression model"
  ],
  "datasets": [
    "Space-Weather Helioseismic and Magnetic Imager Active Region Patch (SHARP) parameters / Helioseismic and Magnetic Imager (HMI) Active Region Patch (HARP) data",
    "Geostationary Operational Environmental Satellites (GOES) data set"
  ],
  "findings": [
    "The mixed LSTM regression model provides detailed information on the exact maximum flux level (intensity) for each flare occurrence rather than just class labels.",
    "Classification models built on top of the

In [82]:
evidence_store = []

for index, paper in enumerate(papers, start=1):
    print(f"Processing paper {index}/{len(papers)}: {paper.title}")

    if not paper.abstract:
        print("⚠️ Skipped: No abstract available.")
        continue

    try:
        evidence = extract_evidence(paper)
        evidence_store.append(evidence)

        print("✅ Evidence extracted.")

    except Exception as e:
        print(f"❌ Failed: {e}")

    # Small pause to reduce API pressure
    time.sleep(3)

print()
print(f"✅ Total evidence items: {len(evidence_store)}")

Processing paper 1/5: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
Gemini temporarily unavailable. Retrying in 10 seconds...
Gemini temporarily unavailable. Retrying in 20 seconds...
✅ Evidence extracted.
Processing paper 2/5: Solar Flare Intensity Prediction With Machine Learning Models
Gemini temporarily unavailable. Retrying in 10 seconds...
Gemini temporarily unavailable. Retrying in 20 seconds...
❌ Failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Processing paper 3/5: Machine learning in solar physics
Gemini temporarily unavailable. Retrying in 10 seconds...
✅ Evidence extracted.
Processing paper 4/5: Review of Solar Energetic Particle Prediction Models
✅ Evidence extracted.
Processing paper 5/5: Predicting Solar Flares with Machine Learning: Investigating Solar Cycle Dependence
Gemini temporaril

In [84]:
for index, evidence in enumerate(evidence_store, start=1):
    print("=" * 90)
    print(f"EVIDENCE #{index}")
    print("=" * 90)

    print(f"Source: {evidence.source_title}")
    print(f"URL: {evidence.source_url}")
    print(f"\nResearch Focus:\n{evidence.research_focus}")

    print("\nMethods:")
    for item in evidence.methods:
        print(f"- {item}")

    print("\nDatasets / Data Sources:")
    for item in evidence.datasets:
        print(f"- {item}")

    print("\nFindings:")
    for item in evidence.findings:
        print(f"- {item}")

    print("\nLimitations:")
    if evidence.limitations:
        for item in evidence.limitations:
            print(f"- {item}")
    else:
        print("- None explicitly identified from the abstract.")

    print(f"\nEvidence Summary:\n{evidence.evidence_summary}")
    print()

EVIDENCE #1
Source: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
URL: https://doi.org/10.1029/2018sw002061

Research Focus:
The present and future role of machine learning in space weather nowcasting and forecasting, including past applications and future challenges.

Methods:

Datasets / Data Sources:

Findings:
- Machine learning applications in space weather forecasting have concentrated primarily on geomagnetic indices, relativistic electrons at geosynchronous orbits, solar flare occurrence, coronal mass ejection propagation time, and solar wind speed.
- Space weather forecasting requires a paradigm shift toward probabilistic approaches focused on reliable uncertainty assessment.
- Space weather forecasting requires combining physics-based and machine learning approaches (gray box modeling).

Limitations:
- None explicitly identified from the abstract.

Evidence Summary:
This review evaluates the role of machine learning in space weather forecastin

In [87]:
test_evidence

EvidenceItem(paper_id='https://openalex.org/W3024706845', source_title='Solar Flare Intensity Prediction With Machine Learning Models', source_url='https://doi.org/10.1029/2020sw002440', research_focus='Predicting the maximum solar flare intensity and solar flare classifications using machine learning models across various advance time windows.', methods=['Mixed long short-term memory (LSTM) regression model', 'Classification models built on top of the LSTM regression model'], datasets=['Space-Weather Helioseismic and Magnetic Imager Active Region Patch (SHARP) parameters / Helioseismic and Magnetic Imager (HMI) Active Region Patch (HARP) data', 'Geostationary Operational Environmental Satellites (GOES) data set'], findings=['The mixed LSTM regression model provides detailed information on the exact maximum flux level (intensity) for each flare occurrence rather than just class labels.', 'Classification models built on top of the regression model achieved better results in solar flare 

In [89]:
existing_ids = {
    item.paper_id
    for item in evidence_store
}

if test_evidence.paper_id not in existing_ids:
    evidence_store.append(test_evidence)
    print("✅ Previously extracted Paper #2 added to evidence store.")
else:
    print("ℹ️ Paper #2 is already in the evidence store.")

print(f"Total evidence items: {len(evidence_store)}")

✅ Previously extracted Paper #2 added to evidence store.
Total evidence items: 4


In [91]:
paper_5 = papers[4]

print(paper_5.title)

Predicting Solar Flares with Machine Learning: Investigating Solar Cycle Dependence


In [97]:
paper_5_evidence = None

try:
    paper_5_evidence = extract_evidence(papers[4])
    print("✅ Paper #5 evidence extracted successfully.")

except Exception as e:
    print(f"⚠️ Paper #5 could not be processed: {e}")

Gemini temporarily unavailable. Retrying in 10 seconds...
Gemini temporarily unavailable. Retrying in 20 seconds...
✅ Paper #5 evidence extracted successfully.


In [99]:
if paper_5_evidence is not None:

    existing_ids = {
        item.paper_id
        for item in evidence_store
    }

    if paper_5_evidence.paper_id not in existing_ids:
        evidence_store.append(paper_5_evidence)
        print("✅ Paper #5 added to evidence store.")

    else:
        print("ℹ️ Paper #5 is already in evidence store.")

else:
    print("⚠️ Paper #5 evidence is not available yet.")

print(f"Total evidence items: {len(evidence_store)}")

✅ Paper #5 added to evidence store.
Total evidence items: 5


In [101]:
if paper_5_evidence is not None:

    existing_ids = {
        item.paper_id
        for item in evidence_store
    }

    if paper_5_evidence.paper_id not in existing_ids:
        evidence_store.append(paper_5_evidence)
        print("✅ Paper #5 added to evidence store.")

    else:
        print("ℹ️ Paper #5 is already in evidence store.")

else:
    print("⚠️ Paper #5 evidence is not available yet.")

print(f"Total evidence items: {len(evidence_store)}")

ℹ️ Paper #5 is already in evidence store.
Total evidence items: 5


## Phase 5 — Evidence Relevance Evaluation

Not every retrieved source is equally relevant to the user's research question.

This stage evaluates each evidence item before it is used for verification
and downstream reasoning.

Responsibilities:

- compare each source with the original research question
- identify highly relevant, partially relevant, and weakly relevant evidence
- explain why a source is relevant or not
- prevent off-topic evidence from influencing later analysis

In [103]:
class RelevanceAssessment(BaseModel):
    paper_id: str | None = None
    source_title: str
    relevance: str
    keep: bool
    reason: str

In [115]:
def evaluate_evidence_relevance(
    user_question: str,
    evidence_items: list
):
    evidence_payload = []

    for item in evidence_items:
        evidence_payload.append({
            "paper_id": item.paper_id,
            "source_title": item.source_title,
            "research_focus": item.research_focus,
            "evidence_summary": item.evidence_summary
        })

    evidence_text = json.dumps(
        evidence_payload,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Evidence Relevance Evaluator of AURA.

Your task is to determine how relevant each research source is
to the user's original research question.

USER QUESTION:

{user_question}

EVIDENCE SOURCES:

{evidence_text}

For each source:

1. Evaluate TOPICAL relevance only.
2. Do not judge scientific quality yet.
3. Use one of:
   - high
   - medium
   - low
4. Set keep=true for sources that should influence downstream analysis.
5. Set keep=false for sources that are too weakly related or off-topic.
6. Give a short reason.

Return ONLY valid JSON as an array.

Example structure:

[
  {{
    "paper_id": "...",
    "source_title": "...",
    "relevance": "high",
    "keep": true,
    "reason": "..."
  }}
]
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=FAST_MODEL
    )

    result = clean_json_response(response.text)

    return [
        RelevanceAssessment(**item)
        for item in result
    ]

In [117]:
relevance_results = evaluate_evidence_relevance(
    question,
    evidence_store
)

relevance_results

Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite


[RelevanceAssessment(paper_id='https://openalex.org/W2920947047', source_title='The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting', relevance='high', keep=True, reason='Provides a direct overview of machine learning applications for solar flare forecasting and discusses overarching challenges and methodological needs.'),
 RelevanceAssessment(paper_id='https://openalex.org/W4384201335', source_title='Machine learning in solar physics', relevance='high', keep=True, reason='Covers the broad application of machine learning and deep learning to process solar observation data specifically for explosive events like solar flares.'),
 RelevanceAssessment(paper_id='https://openalex.org/W4291109640', source_title='Review of Solar Energetic Particle Prediction Models', relevance='medium', keep=True, reason='Focuses on Solar Energetic Particle (SEP) prediction models rather than solar flares directly, but provides valuable methodological insights on prediction modeling 

In [119]:
for index, result in enumerate(relevance_results, start=1):
    print("=" * 90)
    print(f"SOURCE #{index}")
    print(f"Title: {result.source_title}")
    print(f"Relevance: {result.relevance}")
    print(f"Keep: {result.keep}")
    print(f"Reason: {result.reason}")
    print()

SOURCE #1
Title: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
Relevance: high
Keep: True
Reason: Provides a direct overview of machine learning applications for solar flare forecasting and discusses overarching challenges and methodological needs.

SOURCE #2
Title: Machine learning in solar physics
Relevance: high
Keep: True
Reason: Covers the broad application of machine learning and deep learning to process solar observation data specifically for explosive events like solar flares.

SOURCE #3
Title: Review of Solar Energetic Particle Prediction Models
Relevance: medium
Keep: True
Reason: Focuses on Solar Energetic Particle (SEP) prediction models rather than solar flares directly, but provides valuable methodological insights on prediction modeling approaches (physics-based, empirical, ML).

SOURCE #4
Title: Solar Flare Intensity Prediction With Machine Learning Models
Relevance: high
Keep: True
Reason: Directly relevant as it develops machine learni

In [121]:
keep_ids = {
    result.paper_id
    for result in relevance_results
    if result.keep
}

filtered_evidence_store = [
    evidence
    for evidence in evidence_store
    if evidence.paper_id in keep_ids
]

print(f"Original evidence items: {len(evidence_store)}")
print(f"Relevant evidence items: {len(filtered_evidence_store)}")

Original evidence items: 5
Relevant evidence items: 5


In [123]:
core_evidence_store = [
    evidence
    for evidence in evidence_store
    if any(
        result.paper_id == evidence.paper_id
        and result.relevance == "high"
        for result in relevance_results
    )
]

supporting_evidence_store = [
    evidence
    for evidence in evidence_store
    if any(
        result.paper_id == evidence.paper_id
        and result.relevance == "medium"
        for result in relevance_results
    )
]

print(f"Core evidence: {len(core_evidence_store)}")
print(f"Supporting evidence: {len(supporting_evidence_store)}")

Core evidence: 4
Supporting evidence: 1


## Phase 6 — Verification Agent

The Verification Agent checks extracted claims against the original
source material.

Responsibilities:

- verify whether extracted claims are actually supported by the paper abstract
- identify unsupported or overstated claims
- preserve source provenance
- assign evidence-level verification status
- prepare trustworthy claims for downstream analysis

In [125]:
class ClaimVerification(BaseModel):
    claim: str
    status: str
    explanation: str


class PaperVerification(BaseModel):
    paper_id: str | None = None
    source_title: str
    verified_claims: List[ClaimVerification]
    

In [127]:
paper_lookup = {
    paper.paper_id: paper
    for paper in papers
}

In [131]:
def verify_evidence_item(evidence: EvidenceItem):
    paper = paper_lookup.get(evidence.paper_id)

    if paper is None or not paper.abstract:
        return None

    claims = []

    claims.extend(evidence.findings)

    for method in evidence.methods:
        claims.append(f"The paper uses or discusses the method: {method}")

    for dataset in evidence.datasets:
        claims.append(f"The paper uses or discusses the data source: {dataset}")

    claims_text = json.dumps(
        claims,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Verification Agent of AURA.

Your task is to verify extracted claims against the ORIGINAL PAPER ABSTRACT.

Do not use outside knowledge.

PAPER TITLE:
{paper.title}

ORIGINAL ABSTRACT:
{paper.abstract}

CLAIMS TO VERIFY:
{claims_text}

For every claim, assign exactly one status:

- supported
- partially_supported
- unsupported

Rules:

1. "supported" means the abstract directly supports the claim.
2. "partially_supported" means the abstract supports only part of the claim.
3. "unsupported" means the claim cannot be justified from the abstract.
4. Do not infer missing facts.
5. Be strict.

Return ONLY valid JSON:

{{
  "paper_id": "{paper.paper_id}",
  "source_title": "{paper.title}",
  "verified_claims": [
    {{
      "claim": "...",
      "status": "supported",
      "explanation": "..."
    }}
  ]
}}
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=FAST_MODEL
    )

    result = clean_json_response(response.text)

    return PaperVerification(**result)

In [133]:
verification_test = verify_evidence_item(
    core_evidence_store[2]
)

verification_test

Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite


PaperVerification(paper_id='https://openalex.org/W3024706845', source_title='Solar Flare Intensity Prediction With Machine Learning Models', verified_claims=[ClaimVerification(claim='The mixed LSTM regression model provides detailed information on the exact maximum flux level (intensity) for each flare occurrence rather than just class labels.', status='supported', explanation="The abstract states that the model uses 'exact flare intensities instead of class labels recorded in the Geostationary Operational Environmental Satellites (GOES) data set' and offers 'more detailed information about the exact maximum flux level, that is, intensity, for each occurrence of a flare'."), ClaimVerification(claim='Classification models built on top of the regression model achieved better results in solar flare classification compared to Chen et al. (2019).', status='supported', explanation="The abstract explicitly states: 'We also consider classification models built on top of the regression model an

In [135]:
print(
    json.dumps(
        verification_test.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)

{
  "paper_id": "https://openalex.org/W3024706845",
  "source_title": "Solar Flare Intensity Prediction With Machine Learning Models",
  "verified_claims": [
    {
      "claim": "The mixed LSTM regression model provides detailed information on the exact maximum flux level (intensity) for each flare occurrence rather than just class labels.",
      "status": "supported",
      "explanation": "The abstract states that the model uses 'exact flare intensities instead of class labels recorded in the Geostationary Operational Environmental Satellites (GOES) data set' and offers 'more detailed information about the exact maximum flux level, that is, intensity, for each occurrence of a flare'."
    },
    {
      "claim": "Classification models built on top of the regression model achieved better results in solar flare classification compared to Chen et al. (2019).",
      "status": "supported",
      "explanation": "The abstract explicitly states: 'We also consider classification models buil

In [137]:
all_verifications = []

for index, evidence in enumerate(core_evidence_store, start=1):
    print(
        f"Verifying {index}/{len(core_evidence_store)}: "
        f"{evidence.source_title}"
    )

    try:
        verification = verify_evidence_item(evidence)

        if verification is not None:
            all_verifications.append(verification)
            print("✅ Verification completed.")
        else:
            print("⚠️ Verification skipped.")

    except Exception as e:
        print(f"❌ Verification failed: {e}")

    print()

Verifying 1/4: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
✅ Verification completed.

Verifying 2/4: Machine learning in solar physics
Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
✅ Verification completed.

Verifying 3/4: Solar Flare Intensity Prediction With Machine Learning Models
Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
✅ Verification completed.

Verifying 4/4: Predicting Solar Flares with Machine Learning: Investigating Solar Cycle Dependence
Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
✅ Verification completed.



In [139]:
print(f"✅ Verified papers: {len(all_verifications)}")

✅ Verified papers: 4


In [142]:
verification_summary = {
    "supported": 0,
    "partially_supported": 0,
    "unsupported": 0
}

for paper_verification in all_verifications:
    for claim in paper_verification.verified_claims:
        if claim.status in verification_summary:
            verification_summary[claim.status] += 1

print("Verification Summary")
print("--------------------")

for status, count in verification_summary.items():
    print(f"{status}: {count}")

Verification Summary
--------------------
supported: 24
partially_supported: 1
unsupported: 0


In [145]:
trusted_claims = []

for paper_verification in all_verifications:
    for claim in paper_verification.verified_claims:

        if claim.status == "supported":
            trusted_claims.append({
                "paper_id": paper_verification.paper_id,
                "source_title": paper_verification.source_title,
                "claim": claim.claim,
                "status": claim.status,
                "explanation": claim.explanation
            })

print(f"✅ Trusted claims: {len(trusted_claims)}")

✅ Trusted claims: 24


In [148]:
for index, item in enumerate(trusted_claims, start=1):
    print("=" * 90)
    print(f"TRUSTED CLAIM #{index}")
    print(f"Source: {item['source_title']}")
    print(f"Claim: {item['claim']}")
    print()

TRUSTED CLAIM #1
Source: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
Claim: Machine learning applications in space weather forecasting have concentrated primarily on geomagnetic indices, relativistic electrons at geosynchronous orbits, solar flare occurrence, coronal mass ejection propagation time, and solar wind speed.

TRUSTED CLAIM #2
Source: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
Claim: Space weather forecasting requires a paradigm shift toward probabilistic approaches focused on reliable uncertainty assessment.

TRUSTED CLAIM #3
Source: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
Claim: Space weather forecasting requires combining physics-based and machine learning approaches (gray box modeling).

TRUSTED CLAIM #4
Source: Machine learning in solar physics
Claim: Machine learning techniques enable the analysis of large volumes of solar observation data to identify patterns 

In [150]:
review_claims = []

for paper_verification in all_verifications:
    for claim in paper_verification.verified_claims:

        if claim.status != "supported":
            review_claims.append({
                "paper_id": paper_verification.paper_id,
                "source_title": paper_verification.source_title,
                "claim": claim.claim,
                "status": claim.status,
                "explanation": claim.explanation
            })

print(f"Claims requiring review: {len(review_claims)}")

for item in review_claims:
    print("=" * 90)
    print(f"Source: {item['source_title']}")
    print(f"Status: {item['status']}")
    print(f"Claim: {item['claim']}")
    print(f"Reason: {item['explanation']}")

Claims requiring review: 1
Source: Predicting Solar Flares with Machine Learning: Investigating Solar Cycle Dependence
Status: partially_supported
Claim: The paper uses or discusses the method: Binary classification for flare classes >=M (strong), >=C (medium), and >=A (any)
Reason: The abstract mentions predicting whether an active region will produce a flare of class Gamma being >=M, >=C, and >=A, but does not explicitly use the term 'binary classification'.


## Phase 7 — Contradiction & Evidence Relation Detection

This stage compares verified claims across different scientific sources.

Its purpose is to identify:

- supporting evidence across sources
- complementary findings
- tensions caused by different datasets, time periods, or assumptions
- genuine contradictions

Only verified claims are allowed to enter this stage.

In [152]:
claim_records = []

for index, item in enumerate(trusted_claims, start=1):
    claim_records.append({
        "claim_id": f"C{index}",
        "paper_id": item["paper_id"],
        "source_title": item["source_title"],
        "claim": item["claim"]
    })

print(f"Claims prepared for comparison: {len(claim_records)}")

Claims prepared for comparison: 24


In [154]:
class ClaimRelation(BaseModel):
    claim_id_a: str
    claim_id_b: str
    topic: str
    relation: str
    explanation: str


class ContradictionReport(BaseModel):
    relations: List[ClaimRelation] = Field(default_factory=list)
    summary: str

In [156]:
def detect_claim_relations(claim_records: list):

    claims_text = json.dumps(
        claim_records,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Cross-Source Evidence Analysis Agent of AURA.

You are given VERIFIED claims extracted from scientific paper abstracts.

Your task is to compare claims that discuss the same or closely related topics.

CLAIMS:

{claims_text}

For relevant claim pairs, assign exactly one relationship:

- supports
- complements
- tension
- contradicts

Definitions:

supports:
Two independent claims provide similar evidence or conclusions.

complements:
The claims provide different but compatible information about the same topic.

tension:
The claims differ, but the difference may be caused by different datasets,
time periods, prediction targets, assumptions, or experimental settings.

contradicts:
The claims cannot reasonably both be true under the same scope and conditions.

Important rules:

1. Do NOT call something a contradiction just because results differ.
2. Consider differences in datasets, years, targets, and evaluation settings.
3. Only compare claims that meaningfully overlap.
4. Do not use outside knowledge.
5. Do not invent relationships.
6. Return at most the 12 most informative relationships.

Return ONLY valid JSON:

{{
    "relations": [
        {{
            "claim_id_a": "C1",
            "claim_id_b": "C2",
            "topic": "...",
            "relation": "supports",
            "explanation": "..."
        }}
    ],
    "summary": "Overall description of agreement, tension, or contradictions."
}}
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=DEFAULT_MODEL
    )

    result = clean_json_response(response.text)

    return ContradictionReport(**result)

In [158]:
contradiction_report = detect_claim_relations(
    claim_records
)

print(
    json.dumps(
        contradiction_report.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)

Trying model: gemini-3.5-flash
✅ Success with: gemini-3.5-flash
{
  "relations": [
    {
      "claim_id_a": "C15",
      "claim_id_b": "C23",
      "topic": "Solar observation data sources for flare prediction",
      "relation": "supports",
      "explanation": "Both papers utilize the Helioseismic and Magnetic Imager (HMI) active region patch (SHARP/HARP) dataset to study and predict solar flares."
    },
    {
      "claim_id_a": "C16",
      "claim_id_b": "C24",
      "topic": "Solar flare catalogs and datasets",
      "relation": "supports",
      "explanation": "Both papers rely on the Geostationary Operational Environmental Satellites (GOES) datasets/catalogs for flare event labels and intensity verification."
    },
    {
      "claim_id_a": "C13",
      "claim_id_b": "C20",
      "topic": "Machine learning model architectures for solar prediction",
      "relation": "supports",
      "explanation": "Both papers implement and discuss Long Short-Term Memory (LSTM) neural networ

## Phase 8 — Evidence Synthesis & Analysis

This stage synthesizes verified evidence across relevant scientific sources.

Responsibilities:

- identify evidence-backed research findings
- compare methods and data sources
- identify recurring patterns across papers
- preserve disagreements and uncertainty
- distinguish direct evidence from synthesis
- identify evidence gaps

In [160]:
class AnalysisResult(BaseModel):
    key_findings: List[str] = Field(default_factory=list)
    methods_identified: List[str] = Field(default_factory=list)
    data_sources_identified: List[str] = Field(default_factory=list)
    methodological_patterns: List[str] = Field(default_factory=list)
    uncertainties: List[str] = Field(default_factory=list)
    evidence_gaps: List[str] = Field(default_factory=list)
    synthesis: str

In [174]:
def analyze_verified_evidence(
    trusted_claims: list,
    contradiction_report: ContradictionReport
):
    claims_payload = [
        {
            "source_title": item["source_title"],
            "claim": item["claim"]
        }
        for item in trusted_claims
    ]

    relations_payload = [
        relation.model_dump()
        for relation in contradiction_report.relations
    ]

    claims_text = json.dumps(
        claims_payload,
        indent=2,
        ensure_ascii=False
    )

    relations_text = json.dumps(
        relations_payload,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Evidence Synthesis and Analysis Agent of AURA.

Your task is to synthesize VERIFIED scientific evidence.

You may use ONLY the verified claims and cross-source relationships
provided below.

VERIFIED CLAIMS:

{claims_text}

CROSS-SOURCE RELATIONSHIPS:

{relations_text}

Important rules:

1. Do not use outside knowledge.
2. Do not invent methods, datasets, results, or limitations.
3. Do not make claims broader than the evidence supports.
4. Distinguish repeated evidence from general scientific conclusions.
5. If the available evidence does not answer something, record it as an evidence gap.
6. Do not treat absence of evidence as evidence of absence.
7. Do not produce a final implementation recommendation yet.
8. Preserve uncertainty where appropriate.
9. Do not generalize a result from one paper into a universal conclusion.
10. Preserve the scope of each finding, including dataset, model, and time period.
11. Use phrases such as "in one study" or "across the reviewed evidence"
    when the evidence does not support a general conclusion.
12. Do not use words such as "optimal", "best", or "significantly improved"
    unless the verified evidence explicitly supports that wording and scope.

Return ONLY valid JSON:

{{
    "key_findings": [
        "evidence-backed finding"
    ],
    "methods_identified": [
        "method"
    ],
    "data_sources_identified": [
        "data source"
    ],
    "methodological_patterns": [
        "pattern observed across the verified sources"
    ],
    "uncertainties": [
        "uncertainty supported by the evidence"
    ],
    "evidence_gaps": [
        "important question not resolved by the current evidence"
    ],
    "synthesis": "concise evidence-grounded synthesis"
}}
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=REASONING_MODEL
    )

    result = clean_json_response(response.text)

    return AnalysisResult(**result)

In [176]:
analysis_result = analyze_verified_evidence(
    trusted_claims,
    contradiction_report
)

print(
    json.dumps(
        analysis_result.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)

Trying model: gemini-3.8-flash
⚠️ gemini-3.8-flash unavailable or quota exhausted. Trying another model...
Trying model: gemini-3.5-flash
⚠️ gemini-3.5-flash unavailable or quota exhausted. Trying another model...
Trying model: gemini-3.6-flash
⚠️ gemini-3.6-flash unavailable or quota exhausted. Trying another model...
Trying model: gemini-3.7-flash
⚠️ gemini-3.7-flash unavailable or quota exhausted. Trying another model...
Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
{
  "key_findings": [
    "Machine learning applications in space weather forecasting concentrate primarily on geomagnetic indices, relativistic electrons at geosynchronous orbits, solar flare occurrence, coronal mass ejection propagation time, and solar wind speed.",
    "In solar physics, machine learning techniques enable the analysis of large volumes of solar observation data to identify patterns and trends not apparent with traditional methods, automate data analysis, reduce manual labor,

## Phase 9 — Decision & Recommendation Agent

This stage converts verified scientific evidence into a practical,
evidence-grounded technical recommendation.

Responsibilities:

- translate scientific findings into design choices
- recommend methods supported by the reviewed evidence
- identify assumptions and unresolved risks
- distinguish evidence-backed recommendations from engineering choices
- avoid unsupported claims

In [178]:
class DecisionResult(BaseModel):
    recommended_approach: str
    recommended_methods: List[str] = Field(default_factory=list)
    recommended_data_sources: List[str] = Field(default_factory=list)
    evaluation_strategy: List[str] = Field(default_factory=list)
    design_rationale: List[str] = Field(default_factory=list)
    risks_and_uncertainties: List[str] = Field(default_factory=list)
    evidence_gaps: List[str] = Field(default_factory=list)

In [180]:
def create_evidence_based_decision(
    user_question: str,
    analysis_result: AnalysisResult
):
    analysis_text = json.dumps(
        analysis_result.model_dump(),
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Decision and Recommendation Agent of AURA.

Your task is to convert VERIFIED scientific evidence into
a practical technical recommendation.

USER RESEARCH QUESTION:

{user_question}

VERIFIED EVIDENCE ANALYSIS:

{analysis_text}

Important rules:

1. Use ONLY the provided evidence analysis.
2. Do not introduce outside methods, datasets, or results.
3. Do not claim that any method is universally best.
4. Preserve uncertainty and evidence gaps.
5. Separate evidence-backed recommendations from engineering assumptions.
6. Prefer approaches supported by multiple reviewed sources.
7. Preserve source-specific scope.

For example:
If one reviewed study found a 24-hour prediction window useful,
do NOT claim that 24 hours is universally optimal.
State that it was supported by that reviewed study.

8. Do not invent quantitative performance numbers.
9. Do not hide risks or unresolved questions.
10. The recommendation should be suitable for an MVP implementation.

Return ONLY valid JSON:

{{
    "recommended_approach": "concise evidence-grounded technical approach",
    "recommended_methods": [
        "method"
    ],
    "recommended_data_sources": [
        "data source"
    ],
    "evaluation_strategy": [
        "evaluation step"
    ],
    "design_rationale": [
        "evidence-backed rationale"
    ],
    "risks_and_uncertainties": [
        "risk or uncertainty"
    ],
    "evidence_gaps": [
        "unresolved evidence gap"
    ]
}}
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=FAST_MODEL
    )

    result = clean_json_response(response.text)

    return DecisionResult(**result)

In [182]:
decision_result = create_evidence_based_decision(
    question,
    analysis_result
)

print(
    json.dumps(
        decision_result.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)

Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
{
  "recommended_approach": "Build an MVP solar flare prediction system using time-sequence modeling of magnetic parameters from active region patches paired with GOES X-ray flare catalogs, processed via a mixed LSTM regression model and classification layers for a 24-hour prediction window.",
  "recommended_methods": [
    "Machine learning and deep learning approaches",
    "Mixed long short-term memory (LSTM) regression model",
    "Classification models built on top of the LSTM regression model",
    "Time-sequence modeling of 20 magnetic parameters",
    "Evaluation using True Skill Statistic (TSS) and Heidke Skill Score (HSS)",
    "Combinations of physics-based and machine learning approaches (gray box modeling)"
  ],
  "recommended_data_sources": [
    "Space-Weather Helioseismic and Magnetic Imager Active Region Patch (SHARP) parameters / Helioseismic and Magnetic Imager (HMI) Active Region Patch (HARP) 

In [184]:
decision_result.recommended_approach = (
    "Build an MVP solar flare prediction system using time-sequence "
    "modeling of magnetic parameters from active region patches paired "
    "with GOES X-ray flare catalogs, using a mixed LSTM regression model "
    "with classification layers. Use a 24-hour prediction window as an "
    "initial MVP configuration based on one reviewed study, and validate "
    "that choice experimentally."
)

decision_result.design_rationale[1] = (
    "In one reviewed study, SHARP parameters with an LSTM model were "
    "reported as most efficient within 24 hours before prediction time. "
    "AURA therefore treats 24 hours as an initial MVP configuration to "
    "validate, not as a universal optimum."
)

print(
    json.dumps(
        decision_result.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)

{
  "recommended_approach": "Build an MVP solar flare prediction system using time-sequence modeling of magnetic parameters from active region patches paired with GOES X-ray flare catalogs, using a mixed LSTM regression model with classification layers. Use a 24-hour prediction window as an initial MVP configuration based on one reviewed study, and validate that choice experimentally.",
  "recommended_methods": [
    "Machine learning and deep learning approaches",
    "Mixed long short-term memory (LSTM) regression model",
    "Classification models built on top of the LSTM regression model",
    "Time-sequence modeling of 20 magnetic parameters",
    "Evaluation using True Skill Statistic (TSS) and Heidke Skill Score (HSS)",
    "Combinations of physics-based and machine learning approaches (gray box modeling)"
  ],
  "recommended_data_sources": [
    "Space-Weather Helioseismic and Magnetic Imager Active Region Patch (SHARP) parameters / Helioseismic and Magnetic Imager (HMI) Active

## Phase 10 — Implementation Roadmap

This stage converts the evidence-grounded technical decision into
an actionable implementation plan.

Responsibilities:

- break the recommended approach into implementation stages
- define inputs and outputs for each stage
- separate evidence-backed choices from engineering assumptions
- define validation criteria
- preserve unresolved technical decisions

In [186]:
class RoadmapStep(BaseModel):
    step_id: int
    phase: str
    objective: str

    actions: List[str] = Field(default_factory=list)
    inputs: List[str] = Field(default_factory=list)
    outputs: List[str] = Field(default_factory=list)

    evidence_basis: List[str] = Field(default_factory=list)
    engineering_assumptions: List[str] = Field(default_factory=list)

    success_criteria: List[str] = Field(default_factory=list)


class ImplementationRoadmap(BaseModel):
    project_goal: str
    steps: List[RoadmapStep]

    validation_plan: List[str] = Field(default_factory=list)
    unresolved_decisions: List[str] = Field(default_factory=list)

In [188]:
def create_implementation_roadmap(
    user_question: str,
    decision_result: DecisionResult
):
    decision_text = json.dumps(
        decision_result.model_dump(),
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Implementation Planning Agent of AURA.

Your task is to convert an evidence-grounded technical decision into
a practical MVP implementation roadmap.

USER RESEARCH QUESTION:

{user_question}

EVIDENCE-GROUNDED DECISION:

{decision_text}

Important rules:

1. Use the provided decision as the scientific basis.
2. Do not introduce new scientific methods or datasets as if they were
   supported by the reviewed evidence.
3. You may include necessary engineering steps such as data cleaning,
   preprocessing, train/validation/test preparation, model training,
   evaluation, experiment tracking, and documentation.
4. Clearly label such engineering choices as engineering assumptions
   when they are not directly supported by the reviewed evidence.
5. Treat the 24-hour prediction window as an initial configuration
   that must be experimentally validated.
6. Preserve risks related to solar-cycle dependence.
7. Include TSS and HSS in the evaluation plan because they are supported
   by the reviewed evidence.
8. Do not invent performance targets or numerical accuracy thresholds.
9. Keep the roadmap suitable for an MVP.
10. Produce between 5 and 8 implementation steps.

Return ONLY valid JSON:

{{
    "project_goal": "...",
    "steps": [
        {{
            "step_id": 1,
            "phase": "...",
            "objective": "...",
            "actions": [],
            "inputs": [],
            "outputs": [],
            "evidence_basis": [],
            "engineering_assumptions": [],
            "success_criteria": []
        }}
    ],
    "validation_plan": [],
    "unresolved_decisions": []
}}
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=FAST_MODEL
    )

    result = clean_json_response(response.text)

    return ImplementationRoadmap(**result)

In [190]:
roadmap_result = create_implementation_roadmap(
    question,
    decision_result
)

print(
    json.dumps(
        roadmap_result.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)

Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
{
  "project_goal": "Build an MVP solar flare prediction system using time-sequence modeling of magnetic parameters from active region patches paired with GOES X-ray flare catalogs, utilizing a mixed LSTM regression model with classification layers and a 24-hour initial prediction window configuration.",
  "steps": [
    {
      "step_id": 1,
      "phase": "Data Ingestion and Storage",
      "objective": "Ingest and store Space-Weather Helioseismic and Magnetic Imager Active Region Patch (SHARP) / HARP data and GOES X-ray flare catalogs.",
      "actions": [
        "Download historical SHARP parameters and HMI Active Region Patch data spanning June 2010 to December 2018.",
        "Download corresponding GOES X-ray flare catalogs for labeling and target alignment.",
        "Set up local storage and directory structures to organize raw time-series data."
      ],
      "inputs": [
        "SHARP/HARP data sourc

In [192]:
# --- Evidence-scope corrections for the roadmap ---

# Step 2:
# The reviewed evidence mentions 20 magnetic parameters,
# but our current evidence does not identify exactly which 20.
roadmap_result.steps[1].objective = (
    "Prepare the magnetic-parameter time sequences required for the "
    "LSTM pipeline. The exact set of 20 parameters must be confirmed "
    "from the source study or technical documentation."
)

roadmap_result.steps[1].actions[0] = (
    "Identify the exact 20 magnetic parameters from the source study "
    "or dataset documentation before extracting them."
)

# Step 7:
# Avoid claiming that 24 hours will become a universal optimum.
roadmap_result.steps[6].success_criteria = [
    "Produce comparative evidence showing how the initial 24-hour "
    "configuration performs relative to the tested alternative windows.",
    "Document performance variation across different training and testing years."
]

roadmap_result.validation_plan[3] = (
    "Compare the initial 24-hour prediction-window configuration "
    "with alternative window lengths."
)

# Preserve the unresolved parameter-selection issue.
parameter_gap = (
    "Confirm the exact set of 20 magnetic parameters used by the "
    "reviewed LSTM study before reproducing the model."
)

if parameter_gap not in roadmap_result.unresolved_decisions:
    roadmap_result.unresolved_decisions.append(parameter_gap)

print(
    json.dumps(
        roadmap_result.model_dump(),
        indent=2,
        ensure_ascii=False
    )
)

{
  "project_goal": "Build an MVP solar flare prediction system using time-sequence modeling of magnetic parameters from active region patches paired with GOES X-ray flare catalogs, utilizing a mixed LSTM regression model with classification layers and a 24-hour initial prediction window configuration.",
  "steps": [
    {
      "step_id": 1,
      "phase": "Data Ingestion and Storage",
      "objective": "Ingest and store Space-Weather Helioseismic and Magnetic Imager Active Region Patch (SHARP) / HARP data and GOES X-ray flare catalogs.",
      "actions": [
        "Download historical SHARP parameters and HMI Active Region Patch data spanning June 2010 to December 2018.",
        "Download corresponding GOES X-ray flare catalogs for labeling and target alignment.",
        "Set up local storage and directory structures to organize raw time-series data."
      ],
      "inputs": [
        "SHARP/HARP data source (June 2010 to December 2018)",
        "GOES X-ray flare catalogs"
     

## Phase 11 — Final Traceable Research Report

The final stage assembles the verified outputs of AURA into a
traceable research and decision report.

The report is generated from structured pipeline outputs rather than
through a new free-form LLM call.

This helps:

- preserve source provenance
- prevent unsupported claims from being introduced
- connect recommendations back to evidence
- reduce additional model usage and API dependency

In [194]:
source_registry = {}

for index, evidence in enumerate(core_evidence_store, start=1):
    source_id = f"S{index}"

    source_registry[evidence.paper_id] = {
        "source_id": source_id,
        "title": evidence.source_title,
        "url": evidence.source_url
    }

source_registry

{'https://openalex.org/W2920947047': {'source_id': 'S1',
  'title': 'The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting',
  'url': 'https://doi.org/10.1029/2018sw002061'},
 'https://openalex.org/W4384201335': {'source_id': 'S2',
  'title': 'Machine learning in solar physics',
  'url': 'https://doi.org/10.1007/s41116-023-00038-x'},
 'https://openalex.org/W3024706845': {'source_id': 'S3',
  'title': 'Solar Flare Intensity Prediction With Machine Learning Models',
  'url': 'https://doi.org/10.1029/2020sw002440'},
 'https://openalex.org/W2990852342': {'source_id': 'S4',
  'title': 'Predicting Solar Flares with Machine Learning: Investigating Solar Cycle Dependence',
  'url': 'https://doi.org/10.3847/1538-4357/ab89ac'}}

In [196]:
cited_trusted_claims = []

for item in trusted_claims:
    source = source_registry.get(item["paper_id"])

    if source:
        cited_trusted_claims.append({
            "claim": item["claim"],
            "source_id": source["source_id"],
            "source_title": source["title"],
            "source_url": source["url"]
        })

print(f"Cited trusted claims: {len(cited_trusted_claims)}")

Cited trusted claims: 24


In [200]:
def build_final_report(
    question,
    analysis_result,
    decision_result,
    roadmap_result,
    cited_trusted_claims,
    source_registry
):
    lines = []

    lines.append("# AURA — Scientific Research & Decision Report")
    lines.append("")
    lines.append("## Research Question")
    lines.append("")
    lines.append(question.strip())
    lines.append("")

    # Evidence base
    lines.append("## Evidence Base")
    lines.append("")
    lines.append(
        f"- Core scientific sources reviewed: {len(source_registry)}"
    )
    lines.append(
        f"- Verified evidence claims: {len(cited_trusted_claims)}"
    )
    lines.append("")

    # Key findings
    lines.append("## Key Findings")
    lines.append("")

    for finding in analysis_result.key_findings:
        lines.append(f"- {finding}")

    lines.append("")

    # Methods
    lines.append("## Methods Identified")
    lines.append("")

    for method in analysis_result.methods_identified:
        lines.append(f"- {method}")

    lines.append("")

    # Data sources
    lines.append("## Data Sources Identified")
    lines.append("")

    for data_source in analysis_result.data_sources_identified:
        lines.append(f"- {data_source}")

    lines.append("")

    # Recommendation
    lines.append("## Evidence-Grounded Technical Recommendation")
    lines.append("")
    lines.append(decision_result.recommended_approach)
    lines.append("")

    lines.append("### Recommended Methods")
    lines.append("")

    for method in decision_result.recommended_methods:
        lines.append(f"- {method}")

    lines.append("")

    lines.append("### Evaluation Strategy")
    lines.append("")

    for item in decision_result.evaluation_strategy:
        lines.append(f"- {item}")

    lines.append("")

    # Risks
    lines.append("## Risks and Uncertainties")
    lines.append("")

    for risk in decision_result.risks_and_uncertainties:
        lines.append(f"- {risk}")

    lines.append("")

    # Gaps
    lines.append("## Evidence Gaps")
    lines.append("")

    for gap in decision_result.evidence_gaps:
        lines.append(f"- {gap}")

    lines.append("")

    # Roadmap
    lines.append("## Implementation Roadmap")
    lines.append("")

    for step in roadmap_result.steps:
        lines.append(
            f"### Step {step.step_id} — {step.phase}"
        )
        lines.append("")
        lines.append(step.objective)
        lines.append("")

        lines.append("**Actions**")
        for action in step.actions:
            lines.append(f"- {action}")

        lines.append("")
        lines.append("**Evidence Basis**")
        for evidence in step.evidence_basis:
            lines.append(f"- {evidence}")

        lines.append("")
        lines.append("**Engineering Assumptions**")
        for assumption in step.engineering_assumptions:
            lines.append(f"- {assumption}")

        lines.append("")
        lines.append("**Success Criteria**")
        for criterion in step.success_criteria:
            lines.append(f"- {criterion}")

        lines.append("")

    # Unresolved decisions
    lines.append("## Unresolved Decisions")
    lines.append("")

    for item in roadmap_result.unresolved_decisions:
        lines.append(f"- {item}")

    lines.append("")

    # Claim provenance
    lines.append("## Verified Claim Provenance")
    lines.append("")

    for item in cited_trusted_claims:
        lines.append(
            f"- {item['claim']} [{item['source_id']}]"
        )

    lines.append("")

    # References
    lines.append("## References")
    lines.append("")

    ordered_sources = sorted(
        source_registry.values(),
        key=lambda x: int(x["source_id"][1:])
    )

    for source in ordered_sources:
        if source["url"]:
            lines.append(
                f"- [{source['source_id']}] "
                f"{source['title']} — {source['url']}"
            )
        else:
            lines.append(
                f"- [{source['source_id']}] {source['title']}"
            )

    return "\n".join(lines)

In [202]:
final_report = build_final_report(
    question,
    analysis_result,
    decision_result,
    roadmap_result,
    cited_trusted_claims,
    source_registry
)

print(final_report)

# AURA — Scientific Research & Decision Report

## Research Question

I want to build a solar flare prediction system.

Find relevant scientific papers, datasets, machine learning methods,
existing code implementations, and technical documentation.

Compare the approaches, identify their limitations,
and propose an implementation plan.

## Evidence Base

- Core scientific sources reviewed: 4
- Verified evidence claims: 24

## Key Findings

- Machine learning applications in space weather forecasting concentrate primarily on geomagnetic indices, relativistic electrons at geosynchronous orbits, solar flare occurrence, coronal mass ejection propagation time, and solar wind speed.
- In solar physics, machine learning techniques enable the analysis of large volumes of solar observation data to identify patterns and trends not apparent with traditional methods, automate data analysis, reduce manual labor, and increase research efficiency.
- A mixed LSTM regression model provides detailed inf

In [204]:
report_filename = "AURA_solar_flare_research_report.md"

with open(
    report_filename,
    "w",
    encoding="utf-8"
) as file:
    file.write(final_report)

print(f"✅ Final report saved as: {report_filename}")

✅ Final report saved as: AURA_solar_flare_research_report.md


## Current MVP Scope

The current AURA prototype implements an end-to-end,
evidence-grounded workflow for academic paper research.

Implemented:

- research question planning
- academic paper discovery through OpenAlex
- evidence extraction
- relevance evaluation
- claim verification against source abstracts
- cross-source evidence comparison
- evidence synthesis
- technical recommendation
- implementation roadmap
- traceable final report generation

Not yet implemented as independent retrieval pipelines:

- dataset discovery
- code repository discovery
- technical documentation discovery
- full-text paper retrieval and verification
- knowledge graph
- MCP-based tool integration
- ToolGrad / learning loop

Dataset and technical references currently appearing in the report
are derived from the reviewed scientific papers and should not be
interpreted as independently verified external resources.

## Phase 12 — Dataset Discovery Agent

This stage searches for real datasets related to the research question.

Responsibilities:

- use the dataset search task created by the planner
- query an external dataset metadata source
- retrieve real dataset records
- preserve DOI, publisher, year, and source URL
- normalize dataset metadata for downstream analysis

In [206]:
dataset_tasks = [
    task
    for task in plan.tasks
    if task.task_type == "dataset_search"
]

dataset_query = dataset_tasks[0].query

print("Dataset search query:")
print(dataset_query)

Dataset search query:
solar flare benchmark dataset SWAN-SF SDO HMI SHARP parameters GOES X-ray flux


In [208]:
class DatasetSource(BaseModel):
    title: str
    creators: List[str] = Field(default_factory=list)
    year: int | None = None
    publisher: str | None = None
    description: str | None = None
    doi: str | None = None
    url: str | None = None
    resource_type: str | None = None

In [210]:
def search_datacite_datasets(
    query: str,
    limit: int = 5
):
    url = "https://api.datacite.org/dois"

    params = {
        "query": query,
        "resource-type-id": "dataset",
        "page[size]": limit
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    return response.json()

In [212]:
def search_datacite_datasets(
    query: str,
    limit: int = 5
):
    url = "https://api.datacite.org/dois"

    params = {
        "query": query,
        "resource-type-id": "dataset",
        "page[size]": limit
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    return response.json()

In [214]:
def normalize_datacite_datasets(search_result):
    datasets = []

    for item in search_result.get("data", []):
        attributes = item.get("attributes", {})

        titles = attributes.get("titles", [])
        title = (
            titles[0].get("title")
            if titles
            else ""
        )

        creators = []

        for creator in attributes.get("creators", []):
            name = creator.get("name")

            if name:
                creators.append(name)

        descriptions = attributes.get(
            "descriptions",
            []
        )

        description = None

        if descriptions:
            description = descriptions[0].get(
                "description"
            )

        doi = attributes.get("doi")

        dataset = DatasetSource(
            title=title,
            creators=creators,
            year=attributes.get(
                "publicationYear"
            ),
            publisher=attributes.get(
                "publisher"
            ),
            description=description,
            doi=doi,
            url=(
                f"https://doi.org/{doi}"
                if doi
                else attributes.get("url")
            ),
            resource_type=(
                attributes
                .get("types", {})
                .get("resourceTypeGeneral")
            )
        )

        datasets.append(dataset)

    return datasets

In [220]:
print("Current dataset query:")
print(dataset_query)

Current dataset query:
solar flare benchmark dataset SWAN-SF SDO HMI SHARP parameters GOES X-ray flux


In [222]:
dataset_queries = [
    dataset_query,
    "solar flare",
    "solar flare prediction",
    "solar activity",
    "SHARP HMI",
    "GOES solar flare"
]

raw_datasets = None
successful_query = None

for query in dataset_queries:
    print(f"Trying dataset query: {query}")

    result = search_datacite_datasets(
        query=query,
        limit=5
    )

    count = len(result.get("data", []))

    print(f"Datasets returned: {count}")
    print()

    if count > 0:
        raw_datasets = result
        successful_query = query
        break

Trying dataset query: solar flare benchmark dataset SWAN-SF SDO HMI SHARP parameters GOES X-ray flux
Datasets returned: 0

Trying dataset query: solar flare
Datasets returned: 5



In [224]:
if raw_datasets is not None:
    print(f"✅ Dataset search succeeded with: {successful_query}")
else:
    print("⚠️ No relevant datasets found in DataCite.")

✅ Dataset search succeeded with: solar flare


In [226]:
datasets = normalize_datacite_datasets(
    raw_datasets
)

print(f"✅ {len(datasets)} datasets normalized.")

✅ 5 datasets normalized.


In [228]:
for index, dataset in enumerate(datasets, start=1):
    print("=" * 90)
    print(f"DATASET #{index}")
    print(f"Title: {dataset.title}")
    print(f"Year: {dataset.year}")
    print(f"Publisher: {dataset.publisher}")
    print(f"DOI: {dataset.doi}")
    print(f"URL: {dataset.url}")
    print()

DATASET #1
Title: DST/SSODA Ca II K Flare Catalogue v1.1
Year: 2026
Publisher: Zenodo
DOI: 10.5281/zenodo.22846903
URL: https://doi.org/10.5281/zenodo.22846903

DATASET #2
Title: DST/SSODA Ca II K Flare Catalogue v1.1
Year: 2026
Publisher: Zenodo
DOI: 10.5281/zenodo.20710751
URL: https://doi.org/10.5281/zenodo.20710751

DATASET #3
Title: Lightweight and Interpretable Solar Flare Forecasting with Mamba Feature and Temporal Attribution via Integrated Gradients
Year: 2026
Publisher: Zenodo
DOI: 10.5281/zenodo.22748717
URL: https://doi.org/10.5281/zenodo.22748717

DATASET #4
Title: Lightweight and Interpretable Solar Flare Forecasting with Mamba Feature and Temporal Attribution via Integrated Gradients
Year: 2026
Publisher: Zenodo
DOI: 10.5281/zenodo.22748716
URL: https://doi.org/10.5281/zenodo.22748716

DATASET #5
Title: SEP-PRISM Data:   A multi-source dataset for solar energetic particle forecasting
Year: 2026
Publisher: Zenodo
DOI: 10.5281/zenodo.21297634
URL: https://doi.org/10.5281/z

In [230]:
def deduplicate_datasets_by_title(datasets):
    unique_datasets = []
    seen_titles = set()

    for dataset in datasets:
        normalized_title = dataset.title.strip().lower()

        if normalized_title not in seen_titles:
            unique_datasets.append(dataset)
            seen_titles.add(normalized_title)

    return unique_datasets

In [232]:
unique_datasets = deduplicate_datasets_by_title(
    datasets
)

print(f"Original datasets: {len(datasets)}")
print(f"Unique datasets: {len(unique_datasets)}")

Original datasets: 5
Unique datasets: 3


In [234]:
for index, dataset in enumerate(
    unique_datasets,
    start=1
):
    print("=" * 90)
    print(f"DATASET #{index}")
    print(f"Title: {dataset.title}")
    print(f"Year: {dataset.year}")
    print(f"Publisher: {dataset.publisher}")
    print(f"Type: {dataset.resource_type}")
    print(f"DOI: {dataset.doi}")

    if dataset.description:
        print(
            f"Description: "
            f"{dataset.description[:700]}"
        )
    else:
        print("Description: Not available")

    print()

DATASET #1
Title: DST/SSODA Ca II K Flare Catalogue v1.1
Year: 2026
Publisher: Zenodo
Type: Dataset
DOI: 10.5281/zenodo.22846903
Description: The DST/SSODA Ca II K flare catalogue supports the manuscript "A First Calibrated Catalogue of Ca II K Flare Signatures from the Dunn Solar Telescope." It contains 24 retained Ca II K detections: 18 selected by the GOES-guided search and 6 selected by the Ca II K search after masking the dominant GOES peak window. Detection mode describes the selection path; a GOES-independent detection may later be associated with a local GOES feature.

The ZIP contains the 24-row catalogue, a configured-sequence coverage table, SnK class definitions, 24 Ca II K light curves and compact enhancement masks, 21 usable event-window H-alpha light curves, 24 one-minute GOES/XRS-B context curves, per-event region 

DATASET #2
Title: Lightweight and Interpretable Solar Flare Forecasting with Mamba Feature and Temporal Attribution via Integrated Gradients
Year: 2026
Publ

### Dataset Relevance Evaluation

Retrieved dataset records are evaluated against the original research goal.

This stage:

- identifies directly relevant datasets
- separates supporting datasets from off-topic records
- prevents unrelated datasets from entering downstream recommendations
- preserves rejected datasets for traceability

In [238]:
class DatasetRelevanceAssessment(BaseModel):
    title: str
    doi: str | None = None
    relevance: str
    keep: bool
    reason: str

In [240]:
def evaluate_dataset_relevance(
    user_question: str,
    datasets: list
):
    dataset_payload = []

    for dataset in datasets:
        dataset_payload.append({
            "title": dataset.title,
            "year": dataset.year,
            "publisher": dataset.publisher,
            "description": dataset.description,
            "doi": dataset.doi,
            "resource_type": dataset.resource_type
        })

    dataset_text = json.dumps(
        dataset_payload,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Dataset Relevance Evaluator of AURA.

USER RESEARCH QUESTION:

{user_question}

CANDIDATE DATASETS:

{dataset_text}

Evaluate each dataset for its usefulness to the user's
solar flare prediction research goal.

For every dataset:

- relevance must be high, medium, or low
- keep=true only if the dataset could reasonably contribute
  to solar flare prediction research or implementation
- distinguish direct solar flare datasets from related
  space-weather datasets
- do not use outside knowledge
- use only the supplied metadata
- consider whether the dataset appears useful for model
  development, evaluation, or supporting analysis
- if metadata is insufficient, state that explicitly

Return ONLY valid JSON:

[
    {{
        "title": "...",
        "doi": "...",
        "relevance": "high",
        "keep": true,
        "reason": "..."
    }}
]
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=FAST_MODEL
    )

    result = clean_json_response(response.text)

    return [
        DatasetRelevanceAssessment(**item)
        for item in result
    ]

In [242]:
dataset_relevance_results = evaluate_dataset_relevance(
    question,
    unique_datasets
)

for index, result in enumerate(
    dataset_relevance_results,
    start=1
):
    print("=" * 90)
    print(f"DATASET #{index}")
    print(f"Title: {result.title}")
    print(f"Relevance: {result.relevance}")
    print(f"Keep: {result.keep}")
    print(f"Reason: {result.reason}")
    print()

Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
DATASET #1
Title: DST/SSODA Ca II K Flare Catalogue v1.1
Relevance: medium
Keep: True
Reason: Provides a calibrated catalogue of Ca II K flare signatures along with H-alpha light curves and GOES context curves, which can support auxiliary analysis or evaluation of flare signatures, though it is a localized flare detection catalogue rather than a broad forecasting dataset.

DATASET #2
Title: Lightweight and Interpretable Solar Flare Forecasting with Mamba Feature and Temporal Attribution via Integrated Gradients
Relevance: high
Keep: True
Reason: Directly provides ten groups of cross-validation datasets (10CV), comparative datasets (sharp_data_ten_feature.csv, ccmc_waitcompare.csv, sc_waitcompare.csv), experimental code, and pre-trained models specifically designed for solar flare forecasting.

DATASET #3
Title: SEP-PRISM Data:   A multi-source dataset for solar energetic particle forecasting
Relevance: medium
Kee

In [244]:
accepted_dataset_dois = {
    result.doi
    for result in dataset_relevance_results
    if result.keep
}

accepted_datasets = [
    dataset
    for dataset in unique_datasets
    if dataset.doi in accepted_dataset_dois
]

print(f"Candidate datasets: {len(unique_datasets)}")
print(f"Accepted datasets: {len(accepted_datasets)}")

Candidate datasets: 3
Accepted datasets: 3


In [246]:
core_datasets = [
    dataset
    for dataset in unique_datasets
    if any(
        result.doi == dataset.doi
        and result.relevance == "high"
        for result in dataset_relevance_results
    )
]

supporting_datasets = [
    dataset
    for dataset in unique_datasets
    if any(
        result.doi == dataset.doi
        and result.relevance == "medium"
        for result in dataset_relevance_results
    )
]

print(f"Core datasets: {len(core_datasets)}")
print(f"Supporting datasets: {len(supporting_datasets)}")

Core datasets: 1
Supporting datasets: 2


## Phase 13 — Code Discovery Agent

This stage searches for real code implementations related to the
research question.

Responsibilities:

- use the code-search task created by the planner
- retrieve real software repositories
- preserve repository URLs and metadata
- identify implementations directly relevant to solar flare prediction
- separate core implementations from supporting examples

In [248]:
code_tasks = [
    task
    for task in plan.tasks
    if task.task_type == "code_search"
]

code_query = code_tasks[0].query

print("Code search query:")
print(code_query)

Code search query:
solar flare forecasting PyTorch OR TensorFlow GitHub SWAN-SF


In [250]:
class CodeSource(BaseModel):
    name: str
    full_name: str
    description: str | None = None
    url: str
    language: str | None = None
    stars: int = 0
    forks: int = 0
    updated_at: str | None = None

In [252]:
def search_github_repositories(
    query: str,
    limit: int = 5
):
    url = "https://api.github.com/search/repositories"

    params = {
        "q": query,
        "per_page": limit,
        "sort": "stars",
        "order": "desc"
    }

    headers = {
        "Accept": "application/vnd.github+json"
    }

    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    return response.json()

In [254]:
raw_code_results = search_github_repositories(
    query=code_query,
    limit=5
)

print(
    f"Repositories returned: "
    f"{len(raw_code_results.get('items', []))}"
)

Repositories returned: 0


In [262]:
code_queries = [
    code_query,
    "solar flare forecasting",
    "solar flare prediction",
    "solar flare machine learning",
    "solar flare LSTM",
    "SWAN-SF"
]

raw_code_results = None
successful_code_query = None

for query in code_queries:
    print(f"Trying code query: {query}")

    result = search_github_repositories(
        query=query,
        limit=5
    )

    count = len(
        result.get("items", [])
    )

    print(f"Repositories returned: {count}")
    print()

    if count > 0:
        raw_code_results = result
        successful_code_query = query
        break

Trying code query: solar flare forecasting PyTorch OR TensorFlow GitHub SWAN-SF
Repositories returned: 0

Trying code query: solar flare forecasting
Repositories returned: 5



In [264]:
if raw_code_results is not None:
    print(
        f"✅ Code search succeeded with: "
        f"{successful_code_query}"
    )
else:
    print(
        "⚠️ No repositories found with the current queries."
    )

✅ Code search succeeded with: solar flare forecasting


In [266]:
code_sources = normalize_github_repositories(
    raw_code_results
)

print(f"✅ {len(code_sources)} repositories normalized.")

✅ 5 repositories normalized.


In [268]:
for index, repo in enumerate(code_sources, start=1):
    print("=" * 90)
    print(f"REPOSITORY #{index}")
    print(f"Name: {repo.full_name}")
    print(f"Description: {repo.description}")
    print(f"Language: {repo.language}")
    print(f"Stars: {repo.stars}")
    print(f"Forks: {repo.forks}")
    print(f"URL: {repo.url}")
    print()

REPOSITORY #1
Name: USNavalResearchLaboratory/flare_duration_forecasting
Description: Solar Flare Duration Forecasting
Language: Jupyter Notebook
Stars: 8
Forks: 4
URL: https://github.com/USNavalResearchLaboratory/flare_duration_forecasting

REPOSITORY #2
Name: iknyazeva/solar_flares_forecasting
Description: The sun produces solar flares, which have the power to affect the Earth and near-Earth environment with their great bursts of electromagnetic energy and particles. These flares have the power to blow out transformers on power grids and disrupt satellite systems. There is a long lasting task of predictions such events for minimizing its negative impact. Doing so is a difficult task because of the rarity of these events. The success in this task not changes significantly over the last 60 years. Actually, this was a topic of my Ph.D. research, and I did it without any machine learning. But either in the era of big data, there is no big success in this task. The most common approach de

### Code Relevance Evaluation

Retrieved repositories are evaluated against the original research goal.

This stage:

- identifies directly relevant code implementations
- separates supporting repositories from off-topic implementations
- preserves repository provenance
- prevents weakly related code from influencing implementation decisions

In [272]:
class CodeRelevanceAssessment(BaseModel):
    full_name: str
    url: str
    relevance: str
    keep: bool
    reason: str

In [274]:
def evaluate_code_relevance(
    user_question: str,
    repositories: list
):
    repo_payload = []

    for repo in repositories:
        repo_payload.append({
            "full_name": repo.full_name,
            "description": repo.description,
            "language": repo.language,
            "stars": repo.stars,
            "forks": repo.forks,
            "updated_at": repo.updated_at,
            "url": repo.url
        })

    repo_text = json.dumps(
        repo_payload,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Code Relevance Evaluator of AURA.

USER RESEARCH QUESTION:

{user_question}

CANDIDATE CODE REPOSITORIES:

{repo_text}

Evaluate each repository for its usefulness to the user's
solar flare prediction research and implementation goal.

For every repository:

- relevance must be high, medium, or low
- keep=true if the repository could reasonably contribute to
  implementation, comparison, or reproducibility
- distinguish direct solar flare prediction code from related tasks
  such as flare duration prediction
- do not assume code quality from star count alone
- use only the supplied repository metadata
- if metadata is insufficient, say so

Return ONLY valid JSON:

[
    {{
        "full_name": "...",
        "url": "...",
        "relevance": "high",
        "keep": true,
        "reason": "..."
    }}
]
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=FAST_MODEL
    )

    result = clean_json_response(response.text)

    return [
        CodeRelevanceAssessment(**item)
        for item in result
    ]

In [276]:
code_relevance_results = evaluate_code_relevance(
    question,
    code_sources
)

for index, result in enumerate(
    code_relevance_results,
    start=1
):
    print("=" * 90)
    print(f"REPOSITORY #{index}")
    print(f"Name: {result.full_name}")
    print(f"Relevance: {result.relevance}")
    print(f"Keep: {result.keep}")
    print(f"Reason: {result.reason}")
    print()

Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
REPOSITORY #1
Name: USNavalResearchLaboratory/flare_duration_forecasting
Relevance: medium
Keep: True
Reason: Focuses on solar flare duration forecasting rather than occurrence prediction, but provides relevant methodologies, workflows, and domain-specific code implementation from a recognized research lab.

REPOSITORY #2
Name: iknyazeva/solar_flares_forecasting
Relevance: high
Keep: True
Reason: Directly targets solar flare forecasting and addresses key challenges such as time-dependence in features and handling data imbalances (building on standard literature like Bobra et al., 2014), making it highly useful for architectural design and comparison.

REPOSITORY #3
Name: inaf-oact-ai/solar-flare-forecaster
Relevance: high
Keep: True
Reason: Provides a direct solar flare forecasting application leveraging modern transformer models, which is valuable for implementing advanced machine learning methods for time-serie

In [278]:
core_code_sources = [
    repo
    for repo in code_sources
    if any(
        result.url == repo.url
        and result.relevance == "high"
        for result in code_relevance_results
    )
]

supporting_code_sources = [
    repo
    for repo in code_sources
    if any(
        result.url == repo.url
        and result.relevance == "medium"
        for result in code_relevance_results
    )
]

print(f"Core code sources: {len(core_code_sources)}")
print(f"Supporting code sources: {len(supporting_code_sources)}")

Core code sources: 3
Supporting code sources: 2


## Phase 14 — Repository Inspection & Verification

This stage inspects the README content of relevant GitHub repositories.

Responsibilities:

- retrieve repository documentation
- verify implementation details against the README
- identify methods, datasets, frameworks, and artifacts
- avoid assumptions based only on repository titles or descriptions
- preserve repository provenance

In [280]:
import base64

In [282]:
def fetch_github_readme(full_name: str):
    url = f"https://api.github.com/repos/{full_name}/readme"

    headers = {
        "Accept": "application/vnd.github+json"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    if response.status_code == 404:
        return None

    response.raise_for_status()

    data = response.json()

    content = data.get("content")

    if not content:
        return None

    decoded = base64.b64decode(
        content
    ).decode(
        "utf-8",
        errors="replace"
    )

    return decoded

In [284]:
core_repo_readmes = {}

for repo in core_code_sources:
    print(f"Fetching README: {repo.full_name}")

    try:
        readme = fetch_github_readme(
            repo.full_name
        )

        core_repo_readmes[repo.full_name] = readme

        if readme:
            print(
                f"✅ README retrieved "
                f"({len(readme)} characters)"
            )
        else:
            print("⚠️ README not available.")

    except Exception as e:
        print(f"❌ Failed: {e}")

    print()

Fetching README: iknyazeva/solar_flares_forecasting
✅ README retrieved (972 characters)

Fetching README: inaf-oact-ai/solar-flare-forecaster
✅ README retrieved (14388 characters)

Fetching README: ulrichw/flare-forecasting
✅ README retrieved (294 characters)



In [286]:
for repo_name, readme in core_repo_readmes.items():
    print("=" * 90)
    print(repo_name)
    print()

    if readme:
        print(readme[:2000])
    else:
        print("README unavailable.")

    print()

iknyazeva/solar_flares_forecasting

# solar_flares_forecasting
The sun produces solar flares, which have the power to affect the Earth and near-Earth environment with their great bursts of electromagnetic energy and particles. These flares have the power to blow out transformers on power grids and disrupt satellite systems. There is a long lasting task of predictions such events for minimizing its negative impact. Doing so is a difficult task because of the rarity of these events. The success in this task not changes significantly over the last 60 years. Actually, this was a topic of my Ph.D. research, and I did it without any machine learning. But either in the era of big data, there is no big success in this task. The most common approach described in the paper Bobra et al., 2014. The main drawback of the approach is ignoring time dependence on features. Here I tried to use knowledge about working with time series in features. All data for this project could be downloaded from the da

In [288]:
class CodeImplementationEvidence(BaseModel):
    full_name: str
    repository_url: str

    implementation_goal: str

    methods: List[str] = Field(
        default_factory=list
    )

    datasets_or_data_sources: List[str] = Field(
        default_factory=list
    )

    frameworks_or_languages: List[str] = Field(
        default_factory=list
    )

    available_artifacts: List[str] = Field(
        default_factory=list
    )

    limitations_or_missing_information: List[str] = Field(
        default_factory=list
    )

    evidence_summary: str

In [290]:
def extract_code_evidence(
    repositories: list,
    readme_store: dict
):
    repo_payload = []

    for repo in repositories:
        readme = readme_store.get(repo.full_name)

        if not readme:
            continue

        repo_payload.append({
            "full_name": repo.full_name,
            "repository_url": repo.url,
            "repository_description": repo.description,
            "readme": readme
        })

    repo_text = json.dumps(
        repo_payload,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Code Implementation Evidence Agent of AURA.

Your task is to extract structured implementation evidence
ONLY from the supplied GitHub repository metadata and README content.

REPOSITORIES:

{repo_text}

Important rules:

1. Do not use outside knowledge.
2. Do not infer methods that are not explicitly documented.
3. Do not infer datasets only from repository names.
4. If implementation details are missing, record that as a limitation.
5. Preserve framework, language, dataset, and artifact names exactly
   when possible.
6. Treat old dependencies or runtime requirements as implementation
   limitations when explicitly documented.
7. Do not evaluate whether one repository is better than another.

Return ONLY valid JSON as an array:

[
    {{
        "full_name": "...",
        "repository_url": "...",
        "implementation_goal": "...",
        "methods": [],
        "datasets_or_data_sources": [],
        "frameworks_or_languages": [],
        "available_artifacts": [],
        "limitations_or_missing_information": [],
        "evidence_summary": "..."
    }}
]
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=FAST_MODEL
    )

    result = clean_json_response(
        response.text
    )

    return [
        CodeImplementationEvidence(**item)
        for item in result
    ]

In [292]:
code_evidence_store = extract_code_evidence(
    core_code_sources,
    core_repo_readmes
)

print(
    f"✅ Code evidence items: "
    f"{len(code_evidence_store)}"
)

Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
✅ Code evidence items: 3


In [294]:
for index, evidence in enumerate(
    code_evidence_store,
    start=1
):
    print("=" * 90)
    print(f"CODE EVIDENCE #{index}")
    print(f"Repository: {evidence.full_name}")
    print(f"Goal: {evidence.implementation_goal}")
    print(f"Methods: {evidence.methods}")
    print(
        f"Data sources: "
        f"{evidence.datasets_or_data_sources}"
    )
    print(
        f"Frameworks/Languages: "
        f"{evidence.frameworks_or_languages}"
    )
    print(
        f"Artifacts: "
        f"{evidence.available_artifacts}"
    )
    print(
        f"Limitations: "
        f"{evidence.limitations_or_missing_information}"
    )
    print(f"Summary: {evidence.evidence_summary}")
    print()

CODE EVIDENCE #1
Repository: iknyazeva/solar_flares_forecasting
Goal: Predictions of solar flare events to minimize negative impact on Earth/near-Earth environment, incorporating time-series features instead of ignoring time dependence.
Methods: ['Time-series feature engineering']
Data sources: ['All data can be downloaded from the data link (unspecified name in README)']
Frameworks/Languages: []
Artifacts: []
Limitations: ['Data must be downloaded from an external data link.', 'Specific programming languages, frameworks, and model weights are not documented in the README.']
Summary: The repository aims to forecast solar flares by utilizing time-series features to address the drawback of ignoring time dependence found in previous approaches (such as Bobra et al., 2014).

CODE EVIDENCE #2
Repository: inaf-oact-ai/solar-flare-forecaster
Goal: Solar flare forecasting application using transformer models across three modalities (images, videos, and time series).
Methods: ['Image forecaster

## Phase 15 — Technical Documentation Discovery

This stage retrieves trusted technical documentation relevant to
the research and implementation pipeline.

Responsibilities:

- identify official technical documentation sources
- retrieve documentation from authoritative providers
- preserve source URLs and provenance
- extract implementation-relevant information
- distinguish official documentation from research papers and code repositories

In [296]:
documentation_tasks = [
    task
    for task in plan.tasks
    if task.task_type == "documentation_search"
]

documentation_query = documentation_tasks[0].query

print("Documentation search query:")
print(documentation_query)

Documentation search query:
SunPy documentation NASA JSOC data access SDO HMI API NOAA SWPC real-time data


In [298]:
class DocumentationSource(BaseModel):
    name: str
    provider: str
    url: str
    topic: str
    content: str | None = None
    retrieval_status: str

In [320]:
documentation_registry = [
    {
        "name": "SunPy Documentation",
        "provider": "SunPy Project",
        "url": "https://docs.sunpy.org/en/stable/",
        "topic": "Solar data access, processing, Python tools, and remote data interfaces"
    },
    {
        "name": "SDO Data Access",
        "provider": "NASA Solar Dynamics Observatory",
        "url": "https://sdo.gsfc.nasa.gov/data/dataaccess.php",
        "topic": "SDO/HMI data access, HMI observations, and science data archives"
    },
    {
        "name": "GOES X-ray Flux",
        "provider": "NOAA Space Weather Prediction Center",
        "url": "https://www.swpc.noaa.gov/products/goes-x-ray-flux",
        "topic": "GOES X-ray measurements, solar flare monitoring, and data access"
    }
]

print(
    f"Documentation sources registered: "
    f"{len(documentation_registry)}"
)

Documentation sources registered: 3


In [302]:
!pip install beautifulsoup4


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [304]:
from bs4 import BeautifulSoup

In [322]:
def fetch_documentation_page(url: str):
    headers = {
        "User-Agent": "AURA Research Agent/0.1"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    for element in soup([
        "script",
        "style",
        "nav",
        "footer"
    ]):
        element.decompose()

    text = soup.get_text(
        separator=" ",
        strip=True
    )

    return text

In [324]:
documentation_sources = []

for source in documentation_registry:
    print(f"Fetching: {source['name']}")

    try:
        content = fetch_documentation_page(
            source["url"]
        )

        documentation_sources.append(
            DocumentationSource(
                name=source["name"],
                provider=source["provider"],
                url=source["url"],
                topic=source["topic"],
                content=content,
                retrieval_status="success"
            )
        )

        print(
            f"✅ Retrieved "
            f"({len(content)} characters)"
        )

    except Exception as e:
        documentation_sources.append(
            DocumentationSource(
                name=source["name"],
                provider=source["provider"],
                url=source["url"],
                topic=source["topic"],
                content=None,
                retrieval_status="failed"
            )
        )

        print(f"❌ Failed: {e}")


    print()

Fetching: SunPy Documentation
✅ Retrieved (1258 characters)

Fetching: SDO Data Access
✅ Retrieved (4644 characters)

Fetching: GOES X-ray Flux
✅ Retrieved (5506 characters)



In [328]:
class DocumentationEvidence(BaseModel):
    source_name: str
    provider: str
    source_url: str

    capabilities: List[str] = Field(default_factory=list)
    data_products: List[str] = Field(default_factory=list)
    access_methods: List[str] = Field(default_factory=list)

    implementation_relevance: List[str] = Field(
        default_factory=list
    )

    limitations_or_missing_information: List[str] = Field(
        default_factory=list
    )

    evidence_summary: str

In [330]:
def extract_documentation_evidence(
    documentation_sources: list
):
    docs_payload = []

    for source in documentation_sources:

        if (
            source.retrieval_status != "success"
            or not source.content
        ):
            continue

        docs_payload.append({
            "source_name": source.name,
            "provider": source.provider,
            "source_url": source.url,
            "topic": source.topic,
            "content": source.content[:12000]
        })

    docs_text = json.dumps(
        docs_payload,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are the Technical Documentation Evidence Agent of AURA.

Extract implementation-relevant technical evidence ONLY from
the supplied official documentation.

DOCUMENTATION:

{docs_text}

Important rules:

1. Do not use outside knowledge.
2. Do not invent APIs, datasets, or capabilities.
3. Preserve names of documented data products and tools.
4. Record missing implementation details as limitations.
5. Focus on information relevant to solar flare research,
   data access, preprocessing, and implementation.

Return ONLY valid JSON as an array:

[
    {{
        "source_name": "...",
        "provider": "...",
        "source_url": "...",
        "capabilities": [],
        "data_products": [],
        "access_methods": [],
        "implementation_relevance": [],
        "limitations_or_missing_information": [],
        "evidence_summary": "..."
    }}
]
"""

    response = generate_with_fallback(
        prompt,
        preferred_model=FAST_MODEL
    )

    result = clean_json_response(
        response.text
    )

    return [
        DocumentationEvidence(**item)
        for item in result
    ]

In [332]:
documentation_evidence_store = (
    extract_documentation_evidence(
        documentation_sources
    )
)

print(
    f"✅ Documentation evidence items: "
    f"{len(documentation_evidence_store)}"
)

Trying model: gemini-3.5-flash-lite
✅ Success with: gemini-3.5-flash-lite
✅ Documentation evidence items: 3


In [334]:
for index, evidence in enumerate(
    documentation_evidence_store,
    start=1
):
    print("=" * 90)
    print(f"DOCUMENTATION EVIDENCE #{index}")
    print(f"Source: {evidence.source_name}")
    print(f"Provider: {evidence.provider}")
    print(f"Capabilities: {evidence.capabilities}")
    print(f"Data products: {evidence.data_products}")
    print(f"Access methods: {evidence.access_methods}")
    print(
        f"Implementation relevance: "
        f"{evidence.implementation_relevance}"
    )
    print(
        f"Limitations: "
        f"{evidence.limitations_or_missing_information}"
    )
    print(f"Summary: {evidence.evidence_summary}")
    print()

DOCUMENTATION EVIDENCE #1
Source: SunPy Documentation
Provider: SunPy Project
Capabilities: ['Searching and downloading data from multiple data providers', 'Data containers for image and time series data', 'Solar coordinate frames and associated transformations']
Data products: []
Access methods: ['Python interface']
Implementation relevance: ['Provides an environment for solar data analysis, coordinate transformations, and multi-provider data retrieval in Python.']
Limitations: ['Specific API method names, function signatures, and exact input/output parameters are not detailed in the source documentation text.']
Summary: SunPy is a community-developed Python environment that provides data containers, coordinate transformations, and a search/download interface for solar data analysis.

DOCUMENTATION EVIDENCE #2
Source: SDO Data Access
Provider: NASA Solar Dynamics Observatory
Capabilities: ['Browsing and downloading AIA, HMI, and EVE images and animations', 'Creating time series plots 

## Phase 16 — Unified Multi-Source Evidence Store

This stage combines evidence from papers, datasets, code repositories,
and official technical documentation into a unified evidence layer.

Important principles:

- preserve the original source type
- preserve source URLs and provenance
- distinguish verified claims from metadata-derived evidence
- distinguish core evidence from supporting evidence
- avoid treating all source types as equally verified

In [336]:
class UnifiedEvidenceItem(BaseModel):
    evidence_id: str

    source_type: str
    source_name: str
    source_url: str | None = None

    role: str
    verification_status: str

    evidence_text: str
    provenance: str

In [338]:
unified_evidence_store = []

evidence_counter = 1


def add_unified_evidence(
    source_type,
    source_name,
    source_url,
    role,
    verification_status,
    evidence_text,
    provenance
):
    global evidence_counter

    item = UnifiedEvidenceItem(
        evidence_id=f"E{evidence_counter}",
        source_type=source_type,
        source_name=source_name,
        source_url=source_url,
        role=role,
        verification_status=verification_status,
        evidence_text=evidence_text,
        provenance=provenance
    )

    unified_evidence_store.append(item)

    evidence_counter += 1

In [340]:
for item in cited_trusted_claims:
    add_unified_evidence(
        source_type="paper",
        source_name=item["source_title"],
        source_url=item["source_url"],
        role="core",
        verification_status="verified_against_abstract",
        evidence_text=item["claim"],
        provenance=f"Verified scientific claim from {item['source_id']}"
    )

In [342]:
for dataset in core_datasets:

    assessment = next(
        (
            result
            for result in dataset_relevance_results
            if result.doi == dataset.doi
        ),
        None
    )

    add_unified_evidence(
        source_type="dataset",
        source_name=dataset.title,
        source_url=dataset.url,
        role="core",
        verification_status="metadata_retrieved",
        evidence_text=dataset.description or dataset.title,
        provenance=(
            "DataCite dataset metadata"
            + (
                f"; relevance reason: {assessment.reason}"
                if assessment
                else ""
            )
        )
    )


for dataset in supporting_datasets:

    assessment = next(
        (
            result
            for result in dataset_relevance_results
            if result.doi == dataset.doi
        ),
        None
    )

    add_unified_evidence(
        source_type="dataset",
        source_name=dataset.title,
        source_url=dataset.url,
        role="supporting",
        verification_status="metadata_retrieved",
        evidence_text=dataset.description or dataset.title,
        provenance=(
            "DataCite dataset metadata"
            + (
                f"; relevance reason: {assessment.reason}"
                if assessment
                else ""
            )
        )
    )

In [344]:
for evidence in code_evidence_store:
    add_unified_evidence(
        source_type="code",
        source_name=evidence.full_name,
        source_url=evidence.repository_url,
        role="core",
        verification_status="verified_against_readme",
        evidence_text=evidence.evidence_summary,
        provenance="GitHub repository README"
    )

In [346]:
for evidence in documentation_evidence_store:
    add_unified_evidence(
        source_type="documentation",
        source_name=evidence.source_name,
        source_url=evidence.source_url,
        role="core",
        verification_status="official_documentation_retrieved",
        evidence_text=evidence.evidence_summary,
        provenance=f"Official documentation from {evidence.provider}"
    )

In [348]:
print(
    f"✅ Unified evidence items: "
    f"{len(unified_evidence_store)}"
)

source_type_counts = {}

for item in unified_evidence_store:
    source_type_counts[item.source_type] = (
        source_type_counts.get(
            item.source_type,
            0
        ) + 1
    )

print()
print("Evidence by source type:")
print("------------------------")

for source_type, count in source_type_counts.items():
    print(f"{source_type}: {count}")

✅ Unified evidence items: 33

Evidence by source type:
------------------------
paper: 24
dataset: 3
code: 3
documentation: 3


In [350]:
for item in unified_evidence_store[:10]:
    print("=" * 90)
    print(f"Evidence ID: {item.evidence_id}")
    print(f"Type: {item.source_type}")
    print(f"Source: {item.source_name}")
    print(f"Role: {item.role}")
    print(
        f"Verification: "
        f"{item.verification_status}"
    )
    print(f"Evidence: {item.evidence_text[:500]}")
    print(f"Provenance: {item.provenance}")
    print()

Evidence ID: E1
Type: paper
Source: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
Role: core
Verification: verified_against_abstract
Evidence: Machine learning applications in space weather forecasting have concentrated primarily on geomagnetic indices, relativistic electrons at geosynchronous orbits, solar flare occurrence, coronal mass ejection propagation time, and solar wind speed.
Provenance: Verified scientific claim from S1

Evidence ID: E2
Type: paper
Source: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
Role: core
Verification: verified_against_abstract
Evidence: Space weather forecasting requires a paradigm shift toward probabilistic approaches focused on reliable uncertainty assessment.
Provenance: Verified scientific claim from S1

Evidence ID: E3
Type: paper
Source: The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting
Role: core
Verification: verified_against_abstract
Evidence: Spac

In [352]:
print(f"Total evidence items: {len(unified_evidence_store)}")
print()

for source_type in [
    "paper",
    "dataset",
    "code",
    "documentation"
]:
    count = sum(
        1
        for item in unified_evidence_store
        if item.source_type == source_type
    )

    print(f"{source_type}: {count}")

Total evidence items: 33

paper: 24
dataset: 3
code: 3
documentation: 3


## Phase 17 — Multi-Source Final Report v2

This stage generates the final AURA report from the unified
multi-source evidence layer.

The report integrates:

- verified scientific paper claims
- independently discovered datasets
- inspected code repositories
- official technical documentation
- evidence-grounded recommendations
- implementation roadmap
- provenance and verification status

No new free-form LLM call is used in this stage.

In [354]:
def build_multisource_report(
    question,
    analysis_result,
    decision_result,
    roadmap_result,
    core_datasets,
    supporting_datasets,
    code_evidence_store,
    documentation_evidence_store,
    unified_evidence_store
):
    lines = []

    lines.append(
        "# AURA — Multi-Source Scientific Research & Decision Report"
    )
    lines.append("")

    # Research question
    lines.append("## Research Question")
    lines.append("")
    lines.append(question.strip())
    lines.append("")

    # Evidence coverage
    lines.append("## Evidence Coverage")
    lines.append("")

    source_counts = {}

    for item in unified_evidence_store:
        source_counts[item.source_type] = (
            source_counts.get(item.source_type, 0) + 1
        )

    lines.append(
        f"- Total unified evidence items: "
        f"{len(unified_evidence_store)}"
    )

    for source_type, count in source_counts.items():
        lines.append(
            f"- {source_type.capitalize()}: {count}"
        )

    lines.append("")

    # Scientific findings
    lines.append("## Verified Scientific Findings")
    lines.append("")

    for finding in analysis_result.key_findings:
        lines.append(f"- {finding}")

    lines.append("")

    # Datasets
    lines.append("## Dataset Resources")
    lines.append("")

    lines.append("### Core Dataset")
    lines.append("")

    for dataset in core_datasets:
        lines.append(f"**{dataset.title}**")
        lines.append("")
        lines.append(
            f"- Publisher: {dataset.publisher}"
        )
        lines.append(
            f"- Year: {dataset.year}"
        )
        lines.append(
            f"- DOI: {dataset.doi}"
        )
        lines.append(
            f"- URL: {dataset.url}"
        )

        if dataset.description:
            lines.append(
                f"- Description: {dataset.description}"
            )

        lines.append("")

    lines.append("### Supporting Datasets")
    lines.append("")

    for dataset in supporting_datasets:
        lines.append(f"**{dataset.title}**")
        lines.append("")
        lines.append(
            f"- Publisher: {dataset.publisher}"
        )
        lines.append(
            f"- DOI: {dataset.doi}"
        )
        lines.append(
            f"- URL: {dataset.url}"
        )

        if dataset.description:
            lines.append(
                f"- Description: {dataset.description}"
            )

        lines.append("")

    # Code
    lines.append("## Code Implementations")
    lines.append("")

    for evidence in code_evidence_store:
        lines.append(
            f"### {evidence.full_name}"
        )
        lines.append("")
        lines.append(
            f"- Repository: {evidence.repository_url}"
        )
        lines.append(
            f"- Goal: {evidence.implementation_goal}"
        )

        lines.append("")
        lines.append("**Methods**")

        for item in evidence.methods:
            lines.append(f"- {item}")

        lines.append("")
        lines.append("**Data Sources**")

        for item in evidence.datasets_or_data_sources:
            lines.append(f"- {item}")

        lines.append("")
        lines.append("**Frameworks / Languages**")

        for item in evidence.frameworks_or_languages:
            lines.append(f"- {item}")

        lines.append("")
        lines.append("**Available Artifacts**")

        for item in evidence.available_artifacts:
            lines.append(f"- {item}")

        lines.append("")
        lines.append("**Limitations**")

        for item in evidence.limitations_or_missing_information:
            lines.append(f"- {item}")

        lines.append("")

    # Documentation
    lines.append("## Official Technical Documentation")
    lines.append("")

    for evidence in documentation_evidence_store:
        lines.append(
            f"### {evidence.source_name}"
        )
        lines.append("")
        lines.append(
            f"- Provider: {evidence.provider}"
        )
        lines.append(
            f"- URL: {evidence.source_url}"
        )

        lines.append("")
        lines.append("**Capabilities**")

        for item in evidence.capabilities:
            lines.append(f"- {item}")

        lines.append("")
        lines.append("**Data Products**")

        for item in evidence.data_products:
            lines.append(f"- {item}")

        lines.append("")
        lines.append("**Access Methods**")

        for item in evidence.access_methods:
            lines.append(f"- {item}")

        lines.append("")
        lines.append("**Implementation Relevance**")

        for item in evidence.implementation_relevance:
            lines.append(f"- {item}")

        lines.append("")

    # Recommendation
    lines.append(
        "## Evidence-Grounded Technical Recommendation"
    )
    lines.append("")
    lines.append(
        decision_result.recommended_approach
    )
    lines.append("")

    lines.append("### Recommended Methods")
    lines.append("")

    for item in decision_result.recommended_methods:
        lines.append(f"- {item}")

    lines.append("")

    lines.append("### Recommended Data Sources")
    lines.append("")

    for item in decision_result.recommended_data_sources:
        lines.append(f"- {item}")

    lines.append("")

    lines.append("### Evaluation Strategy")
    lines.append("")

    for item in decision_result.evaluation_strategy:
        lines.append(f"- {item}")

    lines.append("")

    # Risks
    lines.append("## Risks and Uncertainties")
    lines.append("")

    for item in decision_result.risks_and_uncertainties:
        lines.append(f"- {item}")

    lines.append("")

    # Evidence gaps
    lines.append("## Evidence Gaps")
    lines.append("")

    for item in decision_result.evidence_gaps:
        lines.append(f"- {item}")

    lines.append("")

    # Roadmap
    lines.append("## Implementation Roadmap")
    lines.append("")

    for step in roadmap_result.steps:
        lines.append(
            f"### Step {step.step_id} — {step.phase}"
        )
        lines.append("")
        lines.append(step.objective)
        lines.append("")

        lines.append("**Actions**")

        for action in step.actions:
            lines.append(f"- {action}")

        lines.append("")
        lines.append("**Evidence Basis**")

        for item in step.evidence_basis:
            lines.append(f"- {item}")

        lines.append("")
        lines.append("**Engineering Assumptions**")

        for item in step.engineering_assumptions:
            lines.append(f"- {item}")

        lines.append("")
        lines.append("**Success Criteria**")

        for item in step.success_criteria:
            lines.append(f"- {item}")

        lines.append("")

    # Unresolved
    lines.append("## Unresolved Decisions")
    lines.append("")

    for item in roadmap_result.unresolved_decisions:
        lines.append(f"- {item}")

    lines.append("")

    # Unified provenance
    lines.append("## Unified Evidence Provenance")
    lines.append("")

    for item in unified_evidence_store:
        lines.append(
            f"- [{item.evidence_id}] "
            f"({item.source_type}, {item.verification_status}) "
            f"{item.source_name}: "
            f"{item.evidence_text}"
        )

    return "\n".join(lines)

In [356]:
multisource_report = build_multisource_report(
    question,
    analysis_result,
    decision_result,
    roadmap_result,
    core_datasets,
    supporting_datasets,
    code_evidence_store,
    documentation_evidence_store,
    unified_evidence_store
)

print(multisource_report[:10000])

# AURA — Multi-Source Scientific Research & Decision Report

## Research Question

I want to build a solar flare prediction system.

Find relevant scientific papers, datasets, machine learning methods,
existing code implementations, and technical documentation.

Compare the approaches, identify their limitations,
and propose an implementation plan.

## Evidence Coverage

- Total unified evidence items: 33
- Paper: 24
- Dataset: 3
- Code: 3
- Documentation: 3

## Verified Scientific Findings

- Machine learning applications in space weather forecasting concentrate primarily on geomagnetic indices, relativistic electrons at geosynchronous orbits, solar flare occurrence, coronal mass ejection propagation time, and solar wind speed.
- In solar physics, machine learning techniques enable the analysis of large volumes of solar observation data to identify patterns and trends not apparent with traditional methods, automate data analysis, reduce manual labor, and increase research efficiency.


In [358]:
multisource_report_filename = (
    "AURA_solar_flare_multisource_report.md"
)

with open(
    multisource_report_filename,
    "w",
    encoding="utf-8"
) as file:
    file.write(multisource_report)

print(
    f"✅ Multi-source report saved as: "
    f"{multisource_report_filename}"
)

✅ Multi-source report saved as: AURA_solar_flare_multisource_report.md


## MVP Completion Checkpoint

AURA MVP currently supports:

- structured research planning
- academic paper discovery
- scientific evidence extraction
- relevance filtering
- claim verification
- cross-source evidence comparison
- dataset discovery
- code repository discovery and README inspection
- official technical documentation retrieval
- unified multi-source evidence storage
- evidence-grounded technical recommendations
- implementation roadmap generation
- traceable multi-source report generation

The next development stage is codebase cleanup, modularization,
testing, and GitHub packaging.

In [360]:
print("AURA MVP CHECKPOINT")
print("-------------------")
print(f"Papers / verified claims: {len(cited_trusted_claims)}")
print(f"Core datasets: {len(core_datasets)}")
print(f"Supporting datasets: {len(supporting_datasets)}")
print(f"Code evidence sources: {len(code_evidence_store)}")
print(f"Documentation sources: {len(documentation_evidence_store)}")
print(f"Unified evidence items: {len(unified_evidence_store)}")
print()
print("✅ Multi-source report generated")
print("✅ AURA MVP pipeline completed")

AURA MVP CHECKPOINT
-------------------
Papers / verified claims: 24
Core datasets: 1
Supporting datasets: 2
Code evidence sources: 3
Documentation sources: 3
Unified evidence items: 33

✅ Multi-source report generated
✅ AURA MVP pipeline completed
